# PhiShark2026 — notebook final unique, code complet et résultats

**Un seul fichier à ouvrir et à exécuter : celui-ci.** Tout le code de l'extraction, des partitions, des quatre branches, des fusions, des22combinaisons, des diagnostics, des figures et des analyses finales est intégré dans les cellules. Aucun module src/ ni autre notebook n'est exécuté.

## Choisir le mode dans la première cellule

- **PRESENTATION** (par défaut) : recharger les résultats FULL vérifiés et afficher tableaux, graphiques et démonstration, sans importer les réseaux ni réentraîner. Les cellules de calcul FULL sont visibles et ignorées dans ce mode.
- **FULL_EXPERIMENT** : exécuter le pipeline complet directement dans les cellules, puis l'analyse finale. Ce mode calcule une expérience distincte liée à la signature de ce fichier ; il ne fait pas passer les anciens checkpoints pour ceux d'une nouvelle version.

Les CSV/ZIP originaux, bibliothèques, caches et résultats restent des fichiers de données nécessaires. Les anciens résultats, labels, seuils et checkpoints ne sont pas modifiés par la présentation. Les copies historiques sont des sauvegardes, sans dépendance d'exécution.

**Utilisation :** sélectionner my_env, enregistrer, redémarrer le noyau et Tout exécuter. Ne pas modifier le code pendant le run. Le mode PRESENTATION suffit pour consulter et défendre le projet déjà entraîné. RQ2 reste non démontrée ; Spark reste BLOCKED_RESOURCE tant que les ressources sont insuffisantes.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, importlib.util
import importlib.metadata as package_metadata
MODE = 'PRESENTATION'  # PRESENTATION ou FULL_EXPERIMENT
# OBLIGATOIRE : chemin du fichier réellement ouvert, y compris le suffixe (1) si présent.
# Laisser vide seulement si PHISHARK_NOTEBOOK_FILE est défini par le lanceur de test.
NOTEBOOK_PATH = os.environ.get('PHISHARK_NOTEBOOK_FILE') or globals().get('__vsc_ipynb_file__', '')
def notebook_sources(path):
    document = json.loads(Path(path).read_text(encoding='utf-8'))
    if not isinstance(document.get('cells'), list):
        raise ValueError('Invalid notebook document')
    return [str(''.join(cell['source'])).strip() for cell in document['cells'] if cell.get('cell_type') == 'code' and ''.join(cell['source']).strip()]

def notebook_code_hash(path):
    return hashlib.sha256(json.dumps(notebook_sources(path), ensure_ascii=False).encode()).hexdigest()

def resolve_notebook_identity(explicit_path, executed_bootstrap):
    if not explicit_path or not str(explicit_path).strip():
        raise RuntimeError('Identité inconnue : définir NOTEBOOK_PATH avec le chemin réel, ou PHISHARK_NOTEBOOK_FILE. Aucun nom de fichier ne sera deviné.')
    path = Path(explicit_path).expanduser().resolve(strict=True)
    if path.suffix.lower() != '.ipynb':
        raise ValueError('The declared file must be an ipynb')
    sources = notebook_sources(path)
    if not sources or executed_bootstrap.strip() != sources[0]:
        raise RuntimeError('La première cellule exécutée diffère du fichier déclaré. Enregistrer le bon notebook, y compris son suffixe, puis redémarrer.')
    return dict(path=str(path), code_hash=notebook_code_hash(path), allowed_sources=sources)

def validate_notebook_session(session):
    current = notebook_code_hash(session['path'])
    if current != session['code_hash']:
        raise RuntimeError('Le code enregistré a changé depuis le démarrage. Ne pas signer des cellules en mémoire anciennes : redémarrer et reprendre depuis la première cellule.')
    return current

def validate_execution_source(session, source):
    validate_notebook_session(session)
    if source.strip() and source.strip() not in session['allowed_sources']:
        raise RuntimeError('Cellule non enregistrée dans ce notebook : enregistrer, redémarrer et exécuter dans l’ordre. Aucun fitting avec ce code en mémoire ne sera autorisé.')

def install_execution_guard(session, shell):
    """Input transformers abort execution on mismatch, unlike pre_run_cell callbacks."""

    def guard(lines):
        validate_execution_source(session, ''.join(lines))
        return lines
    previous = getattr(shell, '_phishark_code_guard', None)
    if previous in shell.input_transformers_post:
        shell.input_transformers_post.remove(previous)
    shell.input_transformers_post.append(guard)
    shell._phishark_code_guard = guard
    return guard

_shell = get_ipython()
_executed_bootstrap = _shell.get_parent()['content']['code']
_declared_path = os.environ.get('PHISHARK_NOTEBOOK_FILE') or NOTEBOOK_PATH
_NOTEBOOK_SESSION = resolve_notebook_identity(_declared_path, _executed_bootstrap)
NOTEBOOK_FILE = Path(_NOTEBOOK_SESSION['path'])
NOTEBOOK_NAME = NOTEBOOK_FILE.name
default_workspace = NOTEBOOK_FILE.parent.parent if NOTEBOOK_FILE.parent.name=='notebooks' else NOTEBOOK_FILE.parent
ROOT = default_workspace.resolve()  # Travail sur C:, sans redirection par variable d'environnement.
RAW_DIRECTORY = (ROOT / 'data/raw').resolve()
if (ROOT/'.runtime').is_dir(): sys.path.insert(0,str(ROOT/'.runtime'))
if MODE == 'FULL_EXPERIMENT':
    requirements = {'numpy':'numpy','pandas':'pandas','torch':'torch','PIL':'Pillow','sklearn':'scikit-learn',
        'pyarrow':'pyarrow','psutil':'psutil','tldextract':'tldextract==5.4.0','matplotlib':'matplotlib'}
    missing = [package for module,package in requirements.items() if importlib.util.find_spec(module) is None]
    if missing: raise RuntimeError('Installer les dépendances puis redémarrer le noyau : '+str(missing))
    if package_metadata.version('tldextract')!='5.4.0': raise RuntimeError('Version PSL attendue : tldextract 5.4.0')
if MODE not in ('PRESENTATION','FULL_EXPERIMENT'): raise ValueError('Mode inconnu')
install_execution_guard(_NOTEBOOK_SESSION,_shell)
print('Fichier déclaré et contrôlé :',NOTEBOOK_FILE)
print('Code enregistré :',_NOTEBOOK_SESSION['code_hash'])
print('Mode :',MODE,'; aucune identité ne provient d’un nom de fichier deviné.')


## Partie A — pipeline complet CRISP-DM

Le code ci-dessous est exécuté directement lorsque MODE vaut FULL_EXPERIMENT. En PRESENTATION, passer simplement par Tout exécuter : ces cellules sont ignorées automatiquement et les résultats vérifiés s'affichent dans la partie C.

Les modèles et les paramètres scientifiques du FULL exécuté sont conservés. La nouvelle enveloppe du notebook change sa signature : un FULL explicitement demandé utilise ses propres artefacts, sans modifier la provenance des résultats historiques.


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    from dataclasses import asdict, dataclass, replace
    import hashlib
    import importlib.metadata as package_metadata
    import importlib.util
    import json
    import os
    from pathlib import Path
    import platform
    import random
    import shutil
    import tempfile
    import numpy as np
    import psutil
    import torch
    import gzip
    from html.parser import HTMLParser
    import io
    from pathlib import Path, PurePosixPath
    import re
    import warnings
    import zipfile
    import pandas as pd
    from PIL import Image
    import importlib.metadata
    import ipaddress
    from urllib.parse import urlsplit, urlunsplit
    import tldextract
    from collections import Counter
    from torch.utils.data import Dataset
    from dataclasses import asdict
    import time
    from torch import nn
    from torch.utils.data import DataLoader
    from sklearn.metrics import average_precision_score
    from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix, precision_recall_curve, brier_score_loss
    from dataclasses import replace
    from IPython.display import display, Markdown, Image as NotebookImage

    import gzip
    import hashlib
    from html.parser import HTMLParser
    import io
    import json
    from pathlib import Path, PurePosixPath
    import re
    import warnings
    import zipfile
    import numpy as np
    import pandas as pd
    from PIL import Image
    from collections import Counter
    import torch
    from torch.utils.data import Dataset
    from dataclasses import asdict
    from pathlib import Path
    import time
    import os
    import threading
    import pyarrow as pa
    import psutil
    from torch import nn
    from torch.utils.data import DataLoader
    from sklearn.metrics import average_precision_score
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    MODEL_IDS = ('U1', 'U2', 'U3', 'U4', 'M1', 'M2')
    ROLES = ('train_encoder', 'train_fusion', 'validation', 'calibration', 'test')


### Configuration complète des modes et protocoles

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    @dataclass(frozen=True)
    class Settings:
        mode: str = 'SMOKE_TEST'
        protocol: str = 'grouped'
        seed: int = 42
        samples_per_class: int = 60
        epochs: int = 2
        patience: int = 2
        batch_size: int = 8
        url_length: int = 192
        html_length: int = 192
        vocabulary_limit: int = 4000
        image_size: int = 32
        threads: int = 1
        learning_rate: float = 0.001
        refinement_limit: int = 8
        bootstrap_replicates: int = 30
        split_fractions: tuple = (0.4, 0.15, 0.15, 0.15, 0.15)
        benchmark_fractions: tuple = (0.1, 0.25, 0.5, 1.0)
        benchmark_repeats: int = 1

        @classmethod
        def full(cls, **changes):
            return cls(mode='FULL_EXPERIMENT', samples_per_class=0, epochs=10, batch_size=32, url_length=256, html_length=1024, vocabulary_limit=16000, image_size=96, threads=2, bootstrap_replicates=200, split_fractions=(0.5, 0.15, 0.1, 0.05, 0.2), benchmark_repeats=3, **changes)

        def checked(self):
            if self.mode not in ('SMOKE_TEST', 'FULL_EXPERIMENT') or self.protocol not in ('grouped', 'chronological'):
                raise ValueError('Unsupported mode or protocol')
            if not np.isclose(sum(self.split_fractions), 1) or min(self.split_fractions) <= 0:
                raise ValueError('Invalid partition fractions')
            if self.mode == 'FULL_EXPERIMENT' and self.samples_per_class:
                raise ValueError('FULL cannot silently subsample data')
            return self


### Signer les paramètres et structures

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def digest(value):
        return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


### Calculer une empreinte SHA-256

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def file_hash(path):
        result = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(2 ** 20), b''):
                result.update(block)
        return result.hexdigest()


### Écrire les résultats JSON atomiquement

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def save_json(path, value):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fd, name = tempfile.mkstemp(dir=path.parent, suffix='.tmp')
        try:
            with os.fdopen(fd, 'w', encoding='utf-8') as stream:
                json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False, default=str)
            Path(name).replace(path)
        finally:
            Path(name).unlink(missing_ok=True)


### Relire les états de reprise

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def load_json(path):
        return json.loads(Path(path).read_text(encoding='utf-8'))


### Sauvegarder les checkpoints atomiquement

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def save_checkpoint(path, state):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temp = path.with_suffix('.tmp')
        torch.save(state, temp)
        temp.replace(path)


### Fixer les graines CPU/GPU

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def seed_everything(seed, threads):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.set_num_threads(threads)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


### Détecter les bibliothèques, Java, GPU, RAM et disque

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def dependency_report():
        modules = {'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow', 'PIL': 'Pillow', 'sklearn': 'scikit-learn', 'torch': 'torch', 'matplotlib': 'matplotlib', 'psutil': 'psutil', 'tldextract': 'tldextract', 'pyspark': 'pyspark'}
        versions = {}
        for module, package in modules.items():
            try:
                versions[package] = package_metadata.version(package)
            except package_metadata.PackageNotFoundError:
                versions[package] = None
        missing = [k for k, v in versions.items() if v is None and k != 'pyspark']
        memory = psutil.virtual_memory()
        return dict(core_status='PASS' if not missing else 'FAIL', missing_core=missing, versions=versions, spark_status='AVAILABLE' if versions['pyspark'] and shutil.which('java') else 'NOT_AVAILABLE', java=shutil.which('java'), python=platform.python_version(), platform=platform.platform(), cuda=torch.cuda.is_available(), available_ram_gb=memory.available / 1000000000.0, free_disk_gb=shutil.disk_usage(ROOT).free / 1000000000.0)


### Vérifier les prérequis avant calcul

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def preflight(settings):
        settings.checked()
        report = dependency_report()
        reasons = list(report['missing_core'])
        warnings = []
        csv = RAW_DIRECTORY / 'phishark2026_public_metadata.csv'
        if not csv.is_file():
            reasons.append('Original metadata CSV missing from data/raw')
        if settings.mode == 'FULL_EXPERIMENT' and csv.is_file():
            expected = pd.read_csv(csv, usecols=['archive_filename'])['archive_filename'].astype(str)
            present = {p.name for p in RAW_DIRECTORY.glob('*.zip')}
            missing = sorted(set(expected) - present)
            report['expected_archives'] = int(len(expected))
            report['available_archives'] = int(len(set(expected) & present))
            report['missing_archive_count'] = len(missing)
            report['missing_archive_examples'] = missing[:5]
            # A handful of absent/corrupt archives can be quarantined, but never silently
            # train on a CSV while most ZIPs are absent.
            allowed_missing = max(5, int(len(expected) * 0.001))
            if len(missing) > allowed_missing:
                reasons.append(f'FULL requires raw ZIP corpus: {len(missing)} archives missing (> {allowed_missing} allowed)')
            elif missing:
                warnings.append(f'{len(missing)} absent ZIPs will be quarantined with provenance')
        if settings.mode == 'FULL_EXPERIMENT':
            if report['available_ram_gb'] < 3:
                warnings.append('RAM disponible inferieure a 3 Go : execution autorisee a la demande de l utilisateur ; risque de ralentissement ou de manque de memoire.')
            # Already allocated image caches need not be budgeted a second time.
            config = {k: v for k, v in asdict(settings).items() if k not in ('protocol', 'epochs', 'patience', 'bootstrap_replicates')}
            cached = False
            for complete in (ROOT / 'data/processed').glob('*/complete.json'):
                try:
                    info = load_json(complete)
                    cached = (info['binding']['code'] == source_signature()
                              and digest(info['binding']['config']) == digest(config)
                              and (complete.parent / 'canonical.parquet').is_file())
                except (OSError, ValueError, KeyError):
                    cached = False
                if cached:
                    break
            required_disk = 2 if cached else 6
            report['disk_budget_scope'] = 'remaining artifacts; cache integrity checked by extract' if cached else 'fresh extraction and artifacts'
            report['required_free_disk_gb'] = required_disk
            if report['free_disk_gb'] < required_disk:
                reasons.append(f'FULL requires at least {required_disk} GB free disk ({report["disk_budget_scope"]})')
            if report['spark_status'] != 'AVAILABLE':
                reasons.append('FULL requires PySpark and Java')
        report.update(status='BLOCKED' if reasons else 'PASS', reasons=reasons, warnings=warnings)
        return report


### Provenance du fichier réellement exécuté

La première cellule exige NOTEBOOK_PATH (ou PHISHARK_NOTEBOOK_FILE). Aucun nom original n’est recherché automatiquement. Son code doit correspondre au fichier déclaré, même avec suffixe (1). Les cellules non enregistrées et les changements du code sur disque interrompent l’exécution via un transformateur d’entrée IPython. Enregistrer puis redémarrer après une modification. Les sorties et cellules vides ne modifient pas la signature du code.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def source_signature():
        return validate_notebook_session(_NOTEBOOK_SESSION)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    settings = Settings.full() if MODE=='FULL_EXPERIMENT' else Settings()
    display(pd.Series(asdict(settings)))
    launch_preflight = preflight(settings)
    display(pd.Series(launch_preflight))
    print('Dossier de travail :', ROOT)
    print('Donnees originales :', RAW_DIRECTORY)
    if launch_preflight['status'] != 'PASS':
        raise RuntimeError('PRECONTROLE BLOQUANT : ' + '; '.join(launch_preflight['reasons']) + '. Resoudre ces points puis redemarrer le noyau. Aucun entrainement lance.')


## 1. Business Understanding

La cible est la **réputation de l’URL au moment de l’annotation**, distincte du phishing actif visible et de la qualité de capture. Un 404 ne change pas le label.

**RQ1** : la fusion surpasse-t-elle chaque modalité isolée ? **RQ2** : généralise-t-elle mieux sur des périodes postérieures ? **RQ3** : quel est le coût du preprocessing, de l’entraînement et le débit selon le volume ?

Le classement utilise AP (average precision). Précision, rappel, F1, FPR et FNR sont mesurés aux seuils choisis exactement sur calibration : meilleur F1 et scénario rappel ≥85 %. Le test ne sélectionne aucun modèle ni seuil.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    FEATURE_VERSION = 'raw-json-v1'
    DEFINITIONS = (('dns', 'records', 'count'), ('whois', 'creation_date', 'present'), ('whois', 'expiration_date', 'present'), ('whois', 'registrar', 'present'), ('whois', 'name_servers', 'count'), ('ip_geo', 'ip', 'present'), ('ip_geo', 'country_code', 'present'), ('ports', 'open_ports', 'count'), ('tls_certificate', 'success', 'boolean'), ('tls_certificate', 'issuer', 'present'), ('tls_certificate', 'subject', 'present'), ('tls_certificate', 'subject_alternative_names', 'count'), ('web_page', 'title', 'length'), ('web_page', 'description', 'length'), ('web_page', 'detected_technologies', 'count'), ('web_page', 'outgoing_links', 'count'), ('web_page', 'total_url_count', 'number'), ('redirection', 'redirect_count', 'number'), ('redirection', 'chain', 'count'), ('compliance_files', 'checked_files', 'count'), ('compliance_files', 'found_files', 'count'))
    FEATURES = tuple(('_'.join(item) for item in DEFINITIONS))


    def url_model_text(value):
        """Same transformation for vocab, training, audit and inference; never fetch URL."""
        raw = '' if value is None else str(value).strip()
        if not raw:
            return ''
        try:
            parsed = urlsplit(raw if '://' in raw or raw.startswith('//') else '//'+raw)
            if parsed.netloc:
                return (parsed.netloc + (parsed.path or '/') + ('?' + parsed.query if parsed.query else '')).lower()
        except ValueError:
            pass
        # Malformed input remains observable, but the scheme and fragment are excluded.
        return re.sub(r'^[a-z][a-z0-9+.-]*://', '', raw, flags=re.I).split('#', 1)[0].lower()


### Conserver les nombres valides et les inconnus

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def finite_nonnegative(value):
        try:
            number = float(value)
            return number if np.isfinite(number) and number >= 0 else np.nan
        except (TypeError, ValueError):
            return np.nan


### HTTP zéro reste inconnu

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def normalized_status(value):
        number = finite_nonnegative(value)
        return int(number) if np.isfinite(number) and number.is_integer() and (100 <= number <= 599) else None


### Extraire l’allowlist technique depuis les JSON

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def extract_numeric(data):
        out = []
        for filename, field, operation in DEFINITIONS:
            value = data.get('data/' + filename + '.json', {}).get(field)
            result = np.nan
            if value is not None:
                if operation == 'present':
                    result = float(bool(value))
                elif operation == 'boolean' and str(value).lower() in ('true', 'false', '0', '1'):
                    result = float(str(value).lower() in ('true', '1'))
                elif operation == 'count':
                    if isinstance(value, (list, dict)):
                        result = float(len(value))
                    elif isinstance(value, str) and field == 'name_servers':
                        result = float(len(value.split()))
                elif operation == 'length' and isinstance(value, str):
                    result = float(len(value))
                elif operation == 'number':
                    result = finite_nonnegative(value)
            out.append(result)
        return np.array(out, np.float32)


### Refuser les features non autorisées

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def assert_features(columns):
        if tuple(columns) != FEATURES:
            raise ValueError('Feature allowlist violation')


### Tokeniser le texte visible

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def word_tokens(text):
        return re.findall('\\w+|[^\\w\\s]', str(text).lower())


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    display(Markdown('**Pipeline :** CSV/ZIP → extraction bornée → Parquet → sites/contenus → nouvelles partitions → U1–U4 → M1/M2 → calibration → test commun → inférence locale.'))
    display(pd.DataFrame({'feature_autorisée': FEATURES}))
    print('Date, source, label, ID, chemin, statut HTTP : variables d’audit, absentes des features du modèle.')


## 2. Data Understanding

L’inventaire est calculé depuis le CSV et les fichiers originaux. Les effectifs, périodes, sources et disponibilités ne sont pas supposés connus. Les dates ISO mélangées sont lues explicitement en UTC. Aucune URL n’est ouverte dans un navigateur.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    JSON_LIMIT = 8 * 2 ** 20
    HTML_LIMIT = 4 * 2 ** 20
    Image.MAX_IMAGE_PIXELS = 20000000


### Parseur HTML : texte visible uniquement

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class VisibleText(HTMLParser):

        def __init__(self):
            super().__init__(convert_charrefs=True)
            self.depth = 0
            self.parts = []

        def handle_starttag(self, tag, attrs):
            if tag in ('script', 'style', 'noscript'):
                self.depth += 1

        def handle_endtag(self, tag):
            if tag in ('script', 'style', 'noscript') and self.depth:
                self.depth -= 1

        def handle_data(self, value):
            if not self.depth:
                self.parts.append(value)


### Nettoyer sans exécuter le document

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def visible_text(raw):
        parser = VisibleText()
        parser.feed(raw.decode('utf-8', errors='replace'))
        return ' '.join(' '.join(parser.parts).lower().split())


### Valider le CSV original et ses labels

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def read_metadata(raw_dir):
        frame = pd.read_csv(Path(raw_dir) / 'phishark2026_public_metadata.csv', dtype={'scan_id': str})
        required = {'scan_id', 'label_id', 'label', 'target_url', 'archive_filename', 'scan_created_at'}
        if not required.issubset(frame):
            raise ValueError('Missing raw CSV fields: ' + str(required - set(frame)))
        if frame.scan_id.isna().any() or not frame.scan_id.is_unique:
            raise ValueError('Ambiguous scan IDs')
        if not frame.scan_id.str.fullmatch('[A-Za-z0-9_-]+').all():
            raise ValueError('Unsafe scan IDs')
        if set(frame.label_id) != {0, 1} or not frame.label.eq(frame.label_id.map({0: 'benign', 1: 'phishing'})).all():
            raise ValueError('CSV labels must be benign=0 / phishing=1')
        if not frame.archive_filename.eq(frame.scan_id + '.zip').all():
            raise ValueError('Unsafe archive mapping')
        frame['scan_created_at'] = pd.to_datetime(frame.scan_created_at, format='mixed', utc=True, errors='raise')
        frame['target_url'] = frame.target_url.fillna('').astype(str)
        return frame.sort_values('scan_id').reset_index(drop=True)


### Sécuriser les membres ZIP

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def validate_members(archive):
        members = archive.infolist()
        seen = set()
        total = 0
        if len(members) > 1000:
            raise ValueError('Too many ZIP members')
        for info in members:
            path = PurePosixPath(info.filename)
            if info.filename in seen or path.is_absolute() or '..' in path.parts or (':' in info.filename) or ('\\' in info.filename):
                raise ValueError('Unsafe or duplicated ZIP member')
            if info.file_size > 32 * 2 ** 20:
                raise ValueError('Oversized ZIP member')
            seen.add(info.filename)
            total += info.file_size
        if total > 128 * 2 ** 20:
            raise ValueError('Oversized ZIP payload')


### Lire JSON, HTML gzip et PNG avec bornes et contrôles

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def read_archive(path, image_size):
        data = {}
        errors = []
        html = b''
        text = ''
        html_state = 'missing'
        image_state = 'missing'
        pixels = np.zeros((3, image_size, image_size), np.uint8)
        expected_artifact_checks = []
        with zipfile.ZipFile(path) as archive:
            validate_members(archive)
            names = set(archive.namelist())
            for name in sorted(names):
                if name.endswith('.json') and (name.startswith('data/') or name == 'metadata.json'):
                    try:
                        if archive.getinfo(name).file_size > JSON_LIMIT:
                            raise ValueError('JSON too large')
                        value = json.loads(archive.read(name))
                        if not isinstance(value, dict):
                            raise ValueError('JSON must be an object')
                        data[name] = value
                    except (ValueError, OSError, UnicodeError, zipfile.BadZipFile) as error:
                        errors.append(name + ':' + type(error).__name__)
            metadata = data.get('metadata.json', {})
            artifact_records = metadata.get('artifacts')
            if artifact_records is None:
                artifact_records = {}
            elif not isinstance(artifact_records, dict):
                errors.append('metadata:artifacts_not_object')
                artifact_records = {}
            for entry in artifact_records.values():
                if not isinstance(entry, dict):
                    continue
                name = entry.get('path')
                expected = entry.get('sha256')
                if name in names and expected:
                    matched = hashlib.sha256(archive.read(name)).hexdigest() == expected
                    expected_artifact_checks.append(matched)
                    if not matched:
                        errors.append('artifact_sha256_mismatch:' + name)
            if 'artifacts/page.html.gz' in names:
                try:
                    with archive.open('artifacts/page.html.gz') as member, gzip.GzipFile(fileobj=member) as stream:
                        html = stream.read(HTML_LIMIT + 1)
                    truncated = len(html) > HTML_LIMIT
                    html = html[:HTML_LIMIT]
                    text = visible_text(html)
                    html_state = 'truncated' if truncated else 'ok' if text else 'empty'
                except (OSError, EOFError, ValueError, zipfile.BadZipFile) as error:
                    html_state = 'invalid'
                    errors.append('html:' + type(error).__name__)
            if 'artifacts/screenshot.png' in names:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('error', Image.DecompressionBombWarning)
                        with Image.open(io.BytesIO(archive.read('artifacts/screenshot.png'))) as image:
                            image.load()
                            image = image.convert('RGB').resize((image_size, image_size), Image.Resampling.BILINEAR)
                            pixels = np.asarray(image).transpose(2, 0, 1).copy()
                    image_state = 'ok'
                except (OSError, ValueError, Image.DecompressionBombWarning, Image.DecompressionBombError, zipfile.BadZipFile) as error:
                    image_state = 'invalid'
                    errors.append('image:' + type(error).__name__)
        return dict(data=data, text=text, image=pixels, html_state=html_state, image_state=image_state, archive_state='ok', uniform=bool(np.max(pixels.std(axis=(1, 2))) < 2) if image_state == 'ok' else False, raw_html_hash=hashlib.sha256(html).hexdigest() if html else '', image_hash=hashlib.sha256(pixels.tobytes()).hexdigest() if image_state == 'ok' else '', errors=errors, declared_hash_checks=expected_artifact_checks)


### Calculer sites et locataires via PSL PRIVATE hors ligne

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class SiteIdentity:

        def __init__(self):
            self.extract = tldextract.TLDExtract(suffix_list_urls=(), cache_dir=None, include_psl_private_domains=True)

        def describe(self, url):
            try:
                parsed = urlsplit(str(url))
                host = (parsed.hostname or '').lower().rstrip('.')
                host = host.encode('idna').decode('ascii')
                if not host:
                    return ('', 'unknown', '')
                try:
                    site = ipaddress.ip_address(host).compressed
                    provider = 'ip_literal'
                except ValueError:
                    result = self.extract(host)
                    site = '.'.join((part for part in (result.domain, result.suffix) if part)) if result.suffix else host
                    provider = result.suffix if result.is_private else 'independent_domain'
                    if host.endswith('.godaddysites.com'):
                        site = '.'.join(host.split('.')[-3:])
                        provider = 'godaddysites.com'
                normalized = urlunsplit((parsed.scheme.lower(), parsed.netloc.lower(), parsed.path or '/', parsed.query, ''))
                return (site, provider, normalized)
            except (ValueError, UnicodeError):
                return ('', 'unknown', '')

        @staticmethod
        def provenance():
            snapshot = Path(tldextract.__file__).parent / '.tld_set_snapshot'
            return dict(library='tldextract', version=importlib.metadata.version('tldextract'), snapshot_sha256=file_hash(snapshot), private_suffixes=True, network_updates=False, cache_directory=None, additional_rule='tenant.godaddysites.com')


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    IDENTITY_KEYS = ('site_id', 'normalized_url', 'url_input_hash', 'html_input_hash', 'raw_html_hash', 'image_input_hash', 'encoding_component')


### Signaler un protocole inexploitable

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class NotFeasible(ValueError):
        pass


### Regrouper les sites et entrées exactes

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def connected_groups(frame):
        parent = {scan: scan for scan in frame.scan_id}

        def root(scan):
            while parent[scan] != scan:
                parent[scan] = parent[parent[scan]]
                scan = parent[scan]
            return scan
        for column in IDENTITY_KEYS:
            if column not in frame:
                continue
            first = {}
            for scan, value in zip(frame.scan_id, frame[column].fillna('')):
                if not value:
                    continue
                if value in first:
                    a, b = (root(scan), root(first[value]))
                    if a != b:
                        parent[max(a, b)] = min(a, b)
                else:
                    first[value] = scan
        return frame.scan_id.map(root)


### Vérifier les classes et les chevauchements

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def check_partitions(frame):
        if not frame.scan_id.is_unique or set(frame.split) != set(ROLES):
            raise NotFeasible('Invalid IDs or missing partition')
        for role, part in frame.groupby('split'):
            if set(part.label_id) != {0, 1}:
                raise NotFeasible('Both classes are required in ' + role)
        for key in ('group_id',) + IDENTITY_KEYS:
            if key not in frame:
                continue
            informative = frame.loc[frame[key].notna() & frame[key].ne('')]
            if informative.groupby(key).split.nunique().gt(1).any():
                raise ValueError('Cross-partition identity: ' + key)


### Construire une partition groupée reproductible

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def grouped(frame, settings):
        frame = frame.copy()
        frame['group_id'] = connected_groups(frame)
        counts = frame.groupby(['group_id', 'label_id']).size().unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)
        values = counts.to_numpy(float)
        target = np.array(settings.split_fractions)[:, None] * values.sum(axis=0)
        scale = np.maximum(target, 1)
        rng = np.random.default_rng(settings.seed)
        solution = None
        for _ in range(12):
            totals = np.zeros_like(target)
            allocation = np.empty(len(values), int)
            order = np.lexsort((rng.random(len(values)), -values.sum(axis=1)))
            for index in order:
                delta = (((totals + values[index] - target) / scale) ** 2 - ((totals - target) / scale) ** 2).sum(axis=1)
                role = int(delta.argmin())
                allocation[index] = role
                totals[role] += values[index]
            cost = float(np.square((totals - target) / scale).sum())
            if np.all(totals > 0) and (solution is None or cost < solution[0]):
                solution = (cost, allocation.copy())
        if solution is None:
            raise NotFeasible('Too few independent groups of both classes for five roles')
        mapping = dict(zip(counts.index, [ROLES[k] for k in solution[1]]))
        frame['split'] = frame.group_id.map(mapping)
        check_partitions(frame)
        return (frame, pd.DataFrame(columns=frame.columns), dict(method='grouped', fractions=list(settings.split_fractions)))


### Construire une partition temporelle purgée

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def chronological(frame, settings, boundaries=None):
        frame = frame.copy()
        frame['group_id'] = connected_groups(frame)
        dates = pd.to_datetime(frame.scan_created_at, format='mixed', utc=True, errors='raise')
        if boundaries is None:
            quantiles = np.cumsum(settings.split_fractions)[:-1]
            edges = [dates.quantile(float(q)) for q in quantiles]
        else:
            edges = list(pd.to_datetime(boundaries, format='mixed', utc=True))
        if len(edges) != 4 or any((a >= b for a, b in zip(edges, edges[1:]))):
            raise NotFeasible('Chronological cutoffs are not distinct')
        frame['split'] = [ROLES[int(np.searchsorted(edges, date, side='right'))] for date in dates]
        crossing = frame.groupby('group_id').split.nunique()
        crossing = set(crossing[crossing > 1].index)
        purged = frame.loc[frame.group_id.isin(crossing)].copy()
        kept = frame.loc[~frame.group_id.isin(crossing)].copy()
        check_partitions(kept)
        for left, right in zip(ROLES, ROLES[1:]):
            a = dates.loc[kept.loc[kept.split.eq(left)].index].max()
            b = dates.loc[kept.loc[kept.split.eq(right)].index].min()
            if a >= b:
                raise NotFeasible('Time roles are not strictly ordered')
        return (kept, purged, dict(method='chronological', utc_boundaries=[str(e) for e in edges], purged=len(purged)))


### Choisir le protocole prédéclaré

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def partition(frame, settings):
        return grouped(frame, settings) if settings.protocol == 'grouped' else chronological(frame, settings)


### Ajuster les transformations sur train_encoder uniquement

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def fit_preprocessing(frame, settings):
        if set(frame.split) != {'train_encoder'}:
            raise ValueError('Only train_encoder may fit preprocessing')
        assert_features(FEATURES)
        raw = frame[list(FEATURES)].to_numpy(np.float32)
        median = np.array([np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0 for col in raw.T], np.float32)
        filled = np.where(np.isfinite(raw), raw, median)
        mean = filled.mean(axis=0)
        scale = filled.std(axis=0)
        scale[scale < 1e-06] = 1
        chars = sorted(set(''.join(frame.target_url.map(url_model_text))))
        counts = Counter((token for text in frame.html_text for token in word_tokens(text)))
        vocabulary = sorted(counts, key=lambda item: (-counts[item], item))[:settings.vocabulary_limit - 2]
        return dict(url_vocab={'<PAD>': 0, '<UNK>': 1, **{c: i + 2 for i, c in enumerate(chars)}}, html_vocab={'<PAD>': 0, '<UNK>': 1, **{t: i + 2 for i, t in enumerate(vocabulary)}}, median=median, mean=mean, scale=scale, features=FEATURES, fitted_ids=frame.scan_id.tolist())


### Encoder avec PAD et UNK

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def encode(text, vocabulary, length, word=False):
        sequence = word_tokens(text) if word else list(str(text).lower())
        result = np.zeros(length, np.int64)
        result[:min(length, len(sequence))] = [vocabulary.get(item, 1) for item in sequence[:length]]
        return result


### Déclarer les modalités disponibles

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def availability(frame):
        return np.column_stack((frame.target_url.str.len().gt(0), frame.html_state.isin(('ok', 'truncated')) & frame.html_text.str.len().gt(0), frame.image_state.eq('ok'), np.isfinite(frame[list(FEATURES)].to_numpy(float)).any(axis=1))).astype(np.float32)


### Détecter les collisions après vocabulaire

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def encoded_collisions(frame, preprocess, settings):
        return encoded_input_audit(frame, preprocess, settings)['collisions']


### Dataset aligné : valeurs, label et scan_id

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class ModalDataset(Dataset):

        def __init__(self, frame, preprocess, settings, model, embeddings=None):
            self.frame = frame.reset_index(drop=True)
            self.preprocess = preprocess
            self.settings = settings
            self.model = model
            self.embeddings = embeddings
            if not self.frame.scan_id.is_unique:
                raise ValueError('Dataset scan IDs are not unique')
            if model == 'M2' and (embeddings is None or len(embeddings) != len(frame)):
                raise ValueError('Embedding alignment error')

        def __len__(self):
            return len(self.frame)

        def __getitem__(self, index):
            row = self.frame.iloc[index]
            p = self.preprocess
            s = self.settings
            if self.model == 'U1':
                x = encode(url_model_text(row.target_url), p['url_vocab'], s.url_length)
            elif self.model == 'U2':
                x = encode(row.html_text, p['html_vocab'], s.html_length, True)
            elif self.model == 'U3':
                x = np.load(ROOT / row.image_relpath).astype(np.float32) / 255
            elif self.model == 'U4':
                raw = row[list(FEATURES)].to_numpy(np.float32)
                missing = ~np.isfinite(raw)
                x = np.concatenate(((np.where(missing, p['median'], raw) - p['mean']) / p['scale'], missing)).astype(np.float32)
            elif self.model == 'M2':
                x = np.array(self.embeddings[index], np.float32, copy=True)
            else:
                raise ValueError('Unknown model')
            return (torch.from_numpy(x), torch.tensor(float(row.label_id)), row.scan_id)


### Sélectionner le petit lot avant tout score

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def identity_match(expected, observed):
        if observed is None or not str(observed).strip():
            return None
        return str(expected) == str(observed)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def capture_policy(row, scan):
        metadata = scan['data'].get('metadata.json', {})
        id_match = identity_match(row['scan_id'], metadata.get('scan_id'))
        target_match = identity_match(row['target_url'], metadata.get('target'))
        reasons = []
        critical = []
        if id_match is not True:
            critical.append('scan_id_mismatch' if id_match is False else 'scan_id_unverifiable')
        if target_match is not True:
            critical.append('target_mismatch' if target_match is False else 'target_unverifiable')
        if any((error.startswith('artifact_sha256_mismatch:') for error in scan['errors'])):
            critical.append('artifact_hash_mismatch')
        if scan.get('original_zip_sha_matches') is False:
            critical.append('original_release_zip_hash_mismatch')
        if scan.get('original_zip_sha_matches') is None:
            reasons.append('original_release_checksum_unavailable')
        if scan.get('archive_state', 'ok') != 'ok':
            critical.append(scan['archive_state'])
        if scan['html_state'] not in ('ok', 'truncated'):
            reasons.append('html_' + scan['html_state'])
        if scan['image_state'] != 'ok':
            reasons.append('image_' + scan['image_state'])
        if scan['uniform']:
            reasons.append('uniform_screenshot')
        if scan['errors']:
            reasons.append('capture_errors')
        state = 'quarantine' if critical else 'review' if reasons else 'valid'
        return dict(target_matches_archive=target_match, id_matches_archive=id_match, quality_state=state, quality_reasons=';'.join(critical + reasons), primary_eligible=not critical)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def quality_report(frame):
        return frame.groupby(['label_id', 'source_name', 'quality_state'], dropna=False).size().reset_index(name='n')


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def timestamp_consistency(row, metadata, tolerance_seconds=1e-06):
        results = {}
        for csv_name, archive_name, prefix in [('scan_created_at', 'created_at', 'created'), ('scan_finished_at', 'finished_at', 'finished')]:
            a = pd.to_datetime(row.get(csv_name), utc=True, errors='coerce')
            b = pd.to_datetime(metadata.get(archive_name), utc=True, errors='coerce')
            delta = (a - b).value / 1000000000.0 if pd.notna(a) and pd.notna(b) else None
            results[prefix + '_csv_archive_delta'] = delta
            results[prefix + '_timestamp_agrees'] = None if delta is None else -tolerance_seconds <= delta <= tolerance_seconds
        return results


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def encoded_input_audit(frame,preprocess,settings):
        """Absent/empty PAD is not identity. Nonempty all-UNK inputs are checked."""
        collisions=[]
        counts=[]
        for name,column,vocabulary,length,word in (
            ('U1','target_url',preprocess['url_vocab'],settings.url_length,False),
            ('U2','html_text',preprocess['html_vocab'],settings.html_length,True)):
            buckets={}
            for row in frame.itertuples():
                original=url_model_text(getattr(row,column)) if name=='U1' else getattr(row,column)
                sequence=word_tokens(original) if word else list(str(original).lower())
                missing=(name=='U2' and row.html_state in ('missing','invalid')) or (name=='U1' and not str(original))
                state='absent' if missing else ('empty' if not sequence else 'present')
                value=encode(original,vocabulary,length,word)
                all_unk=bool(sequence) and bool(np.all(value[value!=0]==1))
                counts.append(dict(scan_id=row.scan_id,split=row.split,model=name,state=state,
                                   all_unk=all_unk,truncated=len(sequence)>length))
                if missing or not sequence: continue
                key=digest(value.tolist())
                buckets.setdefault(key,[]).append((row.scan_id,row.split))
            # Merge the WHOLE equivalence class once it spans roles. Reporting only
            # cross-role pairs leaves same-role siblings behind; repartitioning then
            # exposes those siblings as new collisions even with an unchanged vocab.
            for members in buckets.values():
                if len({role for _,role in members})>1:
                    anchor=members[0][0]
                    collisions.extend((scan,anchor,name) for scan,_ in members[1:])
        return dict(collisions=collisions,states=counts)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def selected_metadata(settings):
        frame = read_metadata(RAW_DIRECTORY)
        if settings.samples_per_class:
            frame = frame.groupby('label_id', group_keys=False).sample(n=settings.samples_per_class, random_state=settings.seed)
        return frame.sort_values('scan_id').reset_index(drop=True)


### Inventorier les données originales

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def inventory():
        frame = read_metadata(RAW_DIRECTORY)
        names = {p.name for p in RAW_DIRECTORY.glob('*.zip')}
        missing = frame.loc[~frame.archive_filename.isin(names), 'scan_id']
        return dict(csv_rows=len(frame), unique_ids=int(frame.scan_id.nunique()), zip_files=len(names), missing_zip_count=len(missing), missing_examples=missing.head(10).tolist(), class_counts={str(k): int(v) for k, v in frame.label_id.value_counts().items()}, min_utc=str(frame.scan_created_at.min()), max_utc=str(frame.scan_created_at.max()), source_counts=frame.groupby(['source_name', 'label_id']).size().reset_index(name='n').to_dict('records') if 'source_name' in frame else [])


### Recréer les caches et la table canonique à partir des bruts

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def original_release_manifest(raw):
        path = Path(raw) / 'SHA256_MANIFEST.csv'
        if not path.is_file():
            return ({}, None)
        table = pd.read_csv(path, dtype={'relative_path': str, 'sha256': str})
        if not {'relative_path', 'sha256', 'size_bytes'}.issubset(table):
            raise ValueError('Invalid original release manifest')
        relevant = table.loc[table.relative_path.str.endswith('.zip') | table.relative_path.eq('phishark2026_public_metadata.csv')]
        if relevant.relative_path.duplicated().any():
            raise ValueError('Ambiguous release manifest')
        for name in relevant.relative_path:
            if Path(name).name != name or '\\' in name or ':' in name:
                raise ValueError('Unsafe release manifest entry')
        return (relevant.set_index('relative_path')[['sha256', 'size_bytes']].to_dict('index'), file_hash(path))


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def extract(settings):
        settings.checked()
        check = preflight(settings)
        if check['status'] != 'PASS':
            raise RuntimeError('BLOCKED: ' + '; '.join(check['reasons']))
        rows = selected_metadata(settings)
        raw = RAW_DIRECTORY
        release, release_hash = original_release_manifest(raw)
        metadata_hash = file_hash(raw / 'phishark2026_public_metadata.csv')
        expected_metadata = release.get('phishark2026_public_metadata.csv')
        if expected_metadata and expected_metadata['sha256'] != metadata_hash:
            raise RuntimeError('Original metadata CSV checksum mismatch; human review required before fitting')
        identity = SiteIdentity()
        extraction_config = {k: v for k, v in asdict(settings).items() if k not in ['protocol', 'epochs', 'patience', 'bootstrap_replicates']}
        stat_manifest = [(r.scan_id, (raw / r.archive_filename).stat().st_size, (raw / r.archive_filename).stat().st_mtime_ns) if (raw / r.archive_filename).is_file() else (r.scan_id, None, None) for r in rows.itertuples()]
        binding = dict(metadata_sha256=metadata_hash, original_release_manifest_sha256=release_hash, code=source_signature(), config=extraction_config, raw_file_stats=digest(stat_manifest), psl=identity.provenance(), features=FEATURE_VERSION, libraries=dependency_report()['versions'])
        version = digest(binding)
        folder = ROOT / 'data/processed' / version[:20]
        folder.mkdir(parents=True, exist_ok=True)
        if (folder / 'complete.json').exists():
            complete = load_json(folder / 'complete.json')
            if digest(complete['binding']) != digest(binding) or file_hash(folder / 'canonical.parquet') != complete['canonical_sha256']:
                raise ValueError('Extraction provenance/cache mismatch')
            return (pd.read_parquet(folder / 'canonical.parquet'), complete)
        if settings.mode == 'FULL_EXPERIMENT' and dependency_report()['free_disk_gb'] < 6:
            raise RuntimeError('Fresh FULL extraction requires at least 6 GB free disk; no valid cache matched the current raw inputs.')
        start = time.perf_counter()
        records = []
        for i, row in enumerate(rows.to_dict('records'), 1):
            archive = raw / row['archive_filename']
            try:
                scan = read_archive(archive, settings.image_size)
            except (OSError, ValueError, zipfile.BadZipFile, EOFError, RuntimeError) as error:
                scan = dict(data={}, text='', image=np.zeros((3, settings.image_size, settings.image_size), np.uint8), html_state='invalid', image_state='invalid', uniform=False, raw_html_hash='', image_hash='', errors=[type(error).__name__ + ':' + str(error)[:200]], archive_state='archive_unreadable', declared_hash_checks=[])
            actual_zip_hash = file_hash(archive) if archive.is_file() else None
            expected_zip = release.get(row['archive_filename'])
            scan['original_zip_sha_matches'] = None if expected_zip is None else bool(actual_zip_hash == expected_zip['sha256'] and archive.is_file() and (archive.stat().st_size == expected_zip['size_bytes']))
            site, provider, normalized = identity.describe(row['target_url'])
            image = folder / (row['scan_id'] + '.npy')
            np.save(image, scan['image'])
            metadata = scan['data'].get('metadata.json', {})
            created = pd.to_datetime(row['scan_created_at'], utc=True)
            finished = pd.to_datetime(row.get('scan_finished_at'), utc=True, errors='coerce')
            duration = (finished - created).total_seconds() if pd.notna(finished) else np.nan
            item = dict(scan_id=row['scan_id'], label_id=int(row['label_id']), target_url=row['target_url'], scan_created_at=created.isoformat(), source_name=row.get('source_name', 'unknown'), archive_filename=row['archive_filename'], site_id=site, provider=provider, normalized_url=normalized, uses_https=row['target_url'].lower().startswith('https://'), html_text=scan['text'], html_state=scan['html_state'], image_state=scan['image_state'], uniform=scan['uniform'], image_relpath=image.relative_to(ROOT).as_posix(), image_file_sha256=file_hash(image), zip_sha256=actual_zip_hash, original_zip_sha_matches=scan['original_zip_sha_matches'], raw_html_hash=scan['raw_html_hash'], html_input_hash=digest(word_tokens(scan['text'])[:settings.html_length]) if scan['text'] else '', url_input_hash=digest(url_model_text(row['target_url'])[:settings.url_length]) if row['target_url'] else '', image_input_hash=scan['image_hash'] if not scan['uniform'] else '', http_status=normalized_status(row.get('http_status')), raw_duration=duration, negative_duration=bool(duration < 0) if np.isfinite(duration) else False, html_truncated=len(word_tokens(scan['text'])) > settings.html_length, url_truncated=len(url_model_text(row['target_url'])) > settings.url_length, capture_errors=';'.join(scan['errors']))
            item.update(zip(FEATURES, extract_numeric(scan['data'])))
            records.append(item)
            item.update(capture_policy(row, scan))
            item.update(timestamp_consistency(row, metadata))
            if i % 25 == 0 or i == len(rows):
                print(f'Extraction originale : {i}/{len(rows)}', flush=True)
        frame = pd.DataFrame(records)
        frame.to_parquet(folder / 'canonical.parquet', index=False)
        review = frame[['scan_id', 'label_id', 'source_name', 'http_status', 'html_state', 'image_state', 'uniform', 'target_matches_archive', 'id_matches_archive', 'quality_state', 'quality_reasons', 'primary_eligible', 'capture_errors']]
        review.to_csv(folder / 'capture_review.csv', index=False)
        quality_report(frame).to_csv(folder / 'quality_counts.csv', index=False)
        complete = dict(binding=binding, version=version, canonical_sha256=file_hash(folder / 'canonical.parquet'), seconds=time.perf_counter() - start, processed_rows=len(frame), labels_changed=False, original_csv_rows=len(read_metadata(raw)))
        save_json(folder / 'complete.json', complete)
        return (frame, complete)


### Stabiliser groupes et vocabulaire sans utiliser les scores

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def construct_partitions(frame,settings):
        source=frame.copy(); revisions=[]
        original_labels=source.set_index('scan_id').label_id.sort_index()
        for attempt in range(max(settings.refinement_limit,len(source))+1):
            selected,purged,policy=partition(source,settings)
            preprocess=fit_preprocessing(selected.loc[selected.split.eq('train_encoder')].reset_index(drop=True),settings)
            collisions=encoded_collisions(selected,preprocess,settings)
            if not collisions:
                if not selected.set_index('scan_id').label_id.sort_index().equals(original_labels.loc[selected.scan_id].sort_index()):
                    raise ValueError('Labels changed during grouping')
                check_partitions(selected)
                print(f'Partitions stable after {attempt} refinements; no residual encoded collision',flush=True)
                return selected.reset_index(drop=True),purged,preprocess,dict(policy=policy,refinements=revisions)
            groups=selected.set_index('scan_id').group_id.to_dict(); parent={g:g for g in selected.group_id.unique()}
            def root(g):
                while parent[g]!=g:
                    parent[g]=parent[parent[g]]; g=parent[g]
                return g
            merges=0
            for left,right,model in collisions:
                a,b=root(groups[left]),root(groups[right])
                if a!=b:
                    parent[max(a,b)]=min(a,b); merges+=1
            if not merges:
                raise NotFeasible('Encoded collisions remain but no group can be merged; inspect partition invariants')
            print(f'Partition refinement {attempt+1}: {len(collisions)} encoding links, {merges} group merges; recomputing train-only vocabularies',flush=True)
            # Each pass merges at least one component. n passes are a finite bound.
            # refinement_limit is now a legacy initial budget, not a stop at round 8.
            # Repartition from all raw-derived observations, not from model scores.
            mapping={scan:root(group) for scan,group in groups.items()}
            previous=source['encoding_component'] if 'encoding_component' in source else source.scan_id
            source['encoding_component']=source.scan_id.map(mapping).fillna(previous)
            revisions.append(dict(round=attempt+1,collisions=[list(p) for p in collisions],merges=merges,groups_before=int(selected.group_id.nunique())))
        raise NotFeasible('Partition refinement incomplete')


### Versionner et recharger une préparation indépendante

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def prepare(settings):
        frame, extraction = extract(settings)
        tag = digest(dict(extraction=extraction['version'], settings=asdict(settings), code=source_signature()))[:20]
        run = ROOT / 'artifacts/runs' / settings.mode.lower() / settings.protocol / tag
        run.mkdir(parents=True, exist_ok=True)
        quarantined = frame.loc[~frame.primary_eligible].copy()
        quarantined[['scan_id', 'label_id', 'source_name', 'quality_reasons']].to_csv(run / 'quarantine.csv', index=False)
        quality_report(frame).to_csv(run / 'quality_counts.csv', index=False)
        all_raw_ids = frame[['scan_id', 'label_id']].copy()
        frame = frame.loc[frame.primary_eligible].copy()
        if settings.protocol == 'chronological':
            unknown_or_disagree = frame['created_timestamp_agrees'].ne(True)
            if unknown_or_disagree.any():
                save_json(run / 'status.json', dict(status='NOT_FEASIBLE', reason='CSV/archive creation times disagree or are unverifiable', affected=int(unknown_or_disagree.sum()), dates_modified=False))
                raise NotFeasible('CSV/archive creation times disagree or are unverifiable; temporal interpretation blocked')
        if (run / 'preparation.json').exists():
            info = load_json(run / 'preparation.json')
            split = ROOT / info['split_relpath']
            if file_hash(split) != info['split_sha256']:
                raise ValueError('Saved partitions changed')
            joined = frame.merge(pd.read_parquet(split), on=['scan_id', 'label_id'], validate='one_to_one')
            check_partitions(joined)
            for row in joined.itertuples():
                if file_hash(ROOT / row.image_relpath) != row.image_file_sha256:
                    raise ValueError('Image cache changed')
            if file_hash(run / 'preprocess.json') != info['preprocess_sha256']:
                raise ValueError('Preprocessing changed')
            preprocess = load_json(run / 'preprocess.json')
            for key in ('median', 'mean', 'scale'):
                preprocess[key] = np.asarray(preprocess[key], np.float32)
            if encoded_collisions(joined, preprocess, settings):
                raise ValueError('Residual encoded collisions in cached partitions')
            return dict(settings=settings, frame=joined, preprocess=preprocess, run=run, provenance=info)
        try:
            selected, purged, preprocess, policy = construct_partitions(frame, settings)
        except NotFeasible as error:
            save_json(run / 'status.json', dict(status='NOT_FEASIBLE', reason=str(error), protocol=settings.protocol, dates_unmodified=True))
            raise
        split_folder = ROOT / 'data/splits' / tag
        split_folder.mkdir(parents=True, exist_ok=True)
        columns = ['scan_id', 'label_id', 'split', 'group_id']
        if 'encoding_component' in selected:
            columns.append('encoding_component')
        path = split_folder / 'manifest.parquet'
        selected[columns].to_parquet(path, index=False)
        selected[columns].to_csv(split_folder / 'manifest.csv', index=False)
        purged[['scan_id', 'label_id']].to_csv(split_folder / 'purged.csv', index=False)
        encoded_audit = encoded_input_audit(selected, preprocess, settings)
        if encoded_audit['collisions']:
            raise ValueError('Residual collisions before training')
        pd.DataFrame(encoded_audit['states']).to_csv(run / 'encoded_input_states.csv', index=False)
        save_json(run / 'preprocess.json', {key: value.tolist() if isinstance(value, np.ndarray) else value for key, value in preprocess.items()})
        info = dict(version=tag, mode=settings.mode, protocol=settings.protocol, code=source_signature(), settings=asdict(settings), extraction=extraction, split_relpath=path.relative_to(ROOT).as_posix(), split_sha256=file_hash(path), preprocess_sha256=file_hash(run / 'preprocess.json'), policy=policy, raw_rows=len(all_raw_ids), quarantined_rows=len(quarantined), eligible_rows=len(frame), raw_label_signature=digest(all_raw_ids.to_dict('records')), residual_encoded_collisions=0, class_counts=selected.groupby(['split', 'label_id']).size().reset_index(name='n').to_dict('records'), test_status='newly generated internal benchmark; dataset previously explored, no independent external test')
        save_json(run / 'preparation.json', info)
        selected.groupby(['label_id', 'provider', 'uses_https']).size().reset_index(name='n').to_csv(run / 'collection_bias_summary.csv', index=False)
        return dict(settings=settings, frame=selected, preprocess=preprocess, run=run, provenance=info)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    raw_inventory = inventory()
    display(pd.Series(raw_inventory))
    metadata = read_metadata(RAW_DIRECTORY)
    weekly = metadata.assign(week=metadata.scan_created_at.dt.strftime('%Y-W%W')).groupby(['week', 'label_id']).size().unstack(fill_value=0)
    import matplotlib.pyplot as plt
    ax = weekly.plot.bar(figsize=(10, 3), title='Collecte originale par semaine et classe')
    ax.set_ylabel('Scans')
    plt.tight_layout()
    plt.show()


### Associations CSV globales — audit descriptif hors apprentissage

Recalculées depuis le CSV original ; ne remplacent pas les associations des 21 features JSON sur train. La table fournie en référence est conservée dans artifacts/reference/full_combinations/.


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    from scipy.stats import chi2_contingency

    def csv_collection_associations():
        # Descriptive whole-corpus audit; never used to fit encoders or choose features/thresholds.
        public = pd.read_csv(RAW_DIRECTORY / 'phishark2026_public_metadata.csv', low_memory=False)
        dates = pd.to_datetime(public.scan_created_at, format='mixed', utc=True, errors='coerce')
        public['day'] = dates.dt.strftime('%Y-%m-%d')
        public['month'] = dates.dt.strftime('%Y-%m')
        public['https'] = public.target_url.fillna('').str.lower().str.startswith('https://')
        columns = ('label_source', 'source_name', 'source_group', 'day', 'month', 'https',
            'page_status', 'free_hosting_provider', 'http_status', 'scan_status', 'has_favicon',
            'has_html', 'is_free_hosting_provider', 'hosting_category', 'has_screenshot')
        rows = []
        for name in columns:
            if name not in public:
                rows.append(dict(variable=name, used_in_model='NO_AUDIT_ONLY', status='COLUMN_ABSENT',
                                 observations=len(public), category_count=None, cramers_v=None, missing_count=None))
                continue
            values = public[name].astype('string').fillna('UNKNOWN')
            tab = pd.crosstab(values, public.label_id)
            valid = min(tab.shape) > 1
            v = float(np.sqrt(chi2_contingency(tab, correction=False)[0] /
                              (tab.to_numpy().sum() * min(tab.shape[0]-1, tab.shape[1]-1)))) if valid else None
            rows.append(dict(variable=name, used_in_model='NO_AUDIT_ONLY', status='AUDITED',
                observations=len(public), category_count=len(tab), cramers_v=v, missing_count=int(public[name].isna().sum())))
        result = pd.DataFrame(rows)
        folder = ROOT / 'artifacts' / 'collection_audit'
        folder.mkdir(parents=True, exist_ok=True)
        result.to_csv(folder / 'csv_associations_recomputed.csv', index=False)
        save_json(folder / 'scope.json', dict(rows=len(public), input_sha256=file_hash(RAW_DIRECTORY / 'phishark2026_public_metadata.csv'),
            descriptive_only=True, used_in_model=False, missing_category='UNKNOWN', y='label_id'))
        return result

    csv_associations = csv_collection_associations()
    display(csv_associations)
    print('Associations CSV globales : audit descriptif, distinct des 21 features JSON et hors apprentissage.')


### Risques de collecte à vérifier sur les bruts

HTTPS/hébergement/source/date peuvent refléter les cohortes de collecte. JSON absent n’est pas un zéro observé ; HTTP 0 est inconnu. Les durées négatives, les captures uniformes, erreurs et divergences de cible sont enregistrées. Ces constats n’entraînent aucune relabelisation. L’inventaire ci-dessous portera seulement sur les archives effectivement extraites.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    canonical_raw, extraction = extract(settings)
    quality = canonical_raw.groupby('label_id').agg(scans=('scan_id', 'size'), html_absent=('html_state', lambda x: x.isin(['missing', 'invalid', 'empty']).sum()), image_absente=('image_state', lambda x: x.ne('ok').sum()), uniformes=('uniform', 'sum'), durées_négatives=('negative_duration', 'sum'))
    display(quality)
    display(canonical_raw.groupby(['label_id', 'provider', 'uses_https']).size().rename('scans').to_frame())
    print('Archives traitées :', extraction['processed_rows'], '; mode :', MODE)


## 3. Data Preparation

ZIP/JSON/HTML/PNG sont lus avec limites, chemins contrôlés et empreintes nouvellement calculées. HTML visible sans scripts/styles ; screenshots RGB redimensionnés ; mesures JSON numériques avec NaN et masques. Les bruts ne sont jamais modifiés.

Sites/locataires : snapshot **tldextract 5.4.0**, PRIVATE activé, mises à jour réseau désactivées ; règle GoDaddy explicite. Les URL normalisées et entrées HTML/image exactes non uniformes construisent les groupes. Cela ne prétend pas connaître toutes les familles proches ou campagnes.

Les cinq rôles sont `train_encoder`, `train_fusion`, `validation`, `calibration`, `test`. La réserve de fusion appartient au budget train et reste disjointe du train des encodeurs. Les vocabulaires, médianes et normalisations sont ajustés sur train_encoder seul. Les collisions introduites par UNK sont regroupées avant tout entraînement, sans scores de modèle.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    context = prepare(settings)
    check_partitions(context['frame'])
    display(pd.DataFrame(context['provenance']['class_counts']))
    display(context['provenance']['policy'])
    print('Run nouveau :', context['run'])
    print('PSL :', context['provenance']['extraction']['binding']['psl'])

    save_json(context['run']/'executions'/f'{time.time_ns()}.json',dict(
        notebook_file=str(NOTEBOOK_FILE),code_signature=source_signature(),
        notebook_file_sha256=file_hash(NOTEBOOK_FILE),mode=settings.mode,
        input_guard='saved-code-only; stop on disk or in-memory source change',
        raw_source_signature=context['provenance']['extraction']['binding']['metadata_sha256']))


### Audit des features — avant entraînement (train uniquement)

Mesure de l'association avec la cible et de la disponibilité de **chacune des 21 variables techniques**, ainsi que des variables de collecte qui restent **interdites en entrée**. Les coefficients ne prouvent pas une causalité ni une fuite. On conserve les variables autorisées : aucune suppression automatique fondée sur la corrélation. Les rapports sont sauvegardés et les modalités brutes ne sont pas filtrées selon la performance du test.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    from scipy.stats import chi2_contingency, spearmanr
    from sklearn.metrics import roc_auc_score

    def feature_provenance_audit(context):
        frame = context['frame'].loc[context['frame'].split.eq('train_encoder')].copy()
        out = context['run'] / 'feature_audit'
        out.mkdir(parents=True, exist_ok=True)
        y = frame.label_id.to_numpy(dtype=int)
        records = []
        for name in FEATURES:
            x = pd.to_numeric(frame[name], errors='coerce').to_numpy(float)
            ok = np.isfinite(x)
            valid = (int(ok.sum()) >= 10 and np.unique(y[ok]).size == 2 and np.unique(x[ok]).size > 1)
            auc = float(roc_auc_score(y[ok], x[ok])) if valid else np.nan
            corr = float(spearmanr(x[ok], y[ok]).statistic) if valid else np.nan
            records.append(dict(feature=name, modality='technical_json', used_in_X=True,
              n_valid=int(ok.sum()), missing_benign=int((~ok & (y==0)).sum()),
              missing_phishing=int((~ok & (y==1)).sum()),
              auc_direction_invariant=abs(auc-0.5)*2 if valid else np.nan,
              spearman=corr))
        features=pd.DataFrame(records)
        features.to_csv(out / 'technical_feature_associations_train.csv', index=False)
        correlations=frame[list(FEATURES)].corr(method='spearman', min_periods=50)
        correlations.to_csv(out / 'technical_correlations_train.csv')
        pairs=[]
        for i,a in enumerate(FEATURES):
            for b in FEATURES[i+1:]:
                r=correlations.loc[a,b]
                if np.isfinite(r) and abs(r)>=0.9:
                    pairs.append(dict(feature_a=a,feature_b=b,spearman=r))
        pd.DataFrame(pairs, columns=['feature_a','feature_b','spearman']).to_csv(out / 'collinearity_pairs_train.csv',index=False)
        extra = frame.copy()
        extra['month'] = pd.to_datetime(extra.scan_created_at, utc=True).dt.strftime('%Y-%m')
        categorical = ('source_name','provider','uses_https','http_status','html_state',
                       'image_state','uniform','quality_state','negative_duration','month')
        context_records=[]
        for name in categorical:
            tab=pd.crosstab(extra[name].astype('string').fillna('UNKNOWN'), extra.label_id)
            if tab.shape[0] > 1 and tab.shape[1] == 2:
                chi2=chi2_contingency(tab, correction=False)[0]
                v=float(np.sqrt(chi2/(tab.values.sum()*min(tab.shape[0]-1,tab.shape[1]-1))))
            else:
                v=np.nan
            context_records.append(dict(feature=name, modality='audit_only',used_in_X=False,
                                        cramers_v=v,distinct=int(tab.shape[0])))
        pd.DataFrame(context_records).to_csv(out / 'collection_context_associations_train.csv',index=False)
        # Importantly, forbid provenance/label/date columns from numerical technical inputs.
        assert not any(k in FEATURES for k in ('source_name','source_group','label_source',
                  'label_id','scan_id','scan_created_at','scan_finished_at','provider','http_status'))
        availability_table = frame[['scan_id','label_id','source_name','provider','uses_https',
                                    'html_state','image_state','uniform','quality_state']].copy()
        mask=availability(frame)
        for j,modality in enumerate(('url','html','image','technical')):
            availability_table['has_'+modality] = mask[:,j].astype(bool)
        availability_table.to_csv(out / 'modality_availability_train.csv',index=False)
        return features.sort_values('auc_direction_invariant',ascending=False),pd.DataFrame(context_records).sort_values('cramers_v',ascending=False)

    technical_associations, collection_associations = feature_provenance_audit(context)
    display(technical_associations)
    display(collection_associations)


## 4. Modeling

Quatre architectures compactes, poids neufs : CNN1D caractères URL, CNN1D tokens HTML visibles, CNN2D image et MLP technique. Chaque encodeur fournit 32 valeurs. AdamW, BCE pondérée sur son train, seed fixée, early stopping sur AP validation ; reprise sur checkpoint compatible. GPU détecté automatiquement, sinon CPU.

Toutes les modalités utilisent les mêmes scans. Les absences sont déclarées par masques : PAD pour HTML absent, image nulle déclarée absente, valeurs numériques imputées avec masques. Une capture uniforme reste observée. Le mode court ne démontre pas la convergence.

### Architecture CNN1D pour caractères ou tokens

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class SequenceCNN(nn.Module):

        def __init__(self, vocabulary_size):
            super().__init__()
            self.embedding = nn.Embedding(vocabulary_size, 24, padding_idx=0)
            self.features = nn.Sequential(nn.Conv1d(24, 48, 5, stride=2, padding=2), nn.ReLU(), nn.Conv1d(48, 32, 3, stride=2, padding=1), nn.ReLU(), nn.AdaptiveMaxPool1d(1))
            self.head = nn.Sequential(nn.Dropout(0.25), nn.Linear(32, 1))

        def encode(self, x):
            return self.features(self.embedding(x).transpose(1, 2)).flatten(1)

        def forward(self, x):
            return self.head(self.encode(x)).squeeze(1)


### Architecture CNN2D compacte

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class ScreenshotCNN(nn.Module):

        def __init__(self):
            super().__init__()
            self.features=nn.Sequential(
                nn.Conv2d(3,24,5,stride=2,padding=2),nn.BatchNorm2d(24),nn.ReLU(),
                nn.Conv2d(24,48,3,stride=2,padding=1),nn.BatchNorm2d(48),nn.ReLU(),
                nn.Conv2d(48,64,3,stride=2,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
                nn.AdaptiveAvgPool2d((2,2)))
            self.projection=nn.Sequential(nn.Flatten(),nn.Linear(256,64),nn.ReLU(),
                                          nn.Dropout(0.2),nn.Linear(64,32),nn.ReLU())
            self.head=nn.Linear(32,1)

        def encode(self,x):
            return self.projection(self.features(x))

        def forward(self,x):
            return self.head(self.encode(x)).squeeze(1)


### Architecture MLP pour technique et fusion

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class DenseClassifier(nn.Module):

        def __init__(self, input_size):
            super().__init__()
            self.features = nn.Sequential(nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.25), nn.Linear(64, 32), nn.ReLU())
            self.head = nn.Linear(32, 1)

        def encode(self, x):
            return self.features(x)

        def forward(self, x):
            return self.head(self.encode(x)).squeeze(1)


### Instancier chacun des cinq réseaux entraînables

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def architecture(name, preprocessing):
        if name == 'U1':
            return SequenceCNN(len(preprocessing['url_vocab']))
        if name == 'U2':
            return SequenceCNN(len(preprocessing['html_vocab']))
        if name == 'U3':
            return ScreenshotCNN()
        if name == 'U4':
            return DenseClassifier(2 * len(FEATURES))
        if name == 'M2':
            return DenseClassifier(132)
        raise ValueError('No trainable architecture for ' + name)


### Inférence chronométrée avec contrôle des IDs

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def inference(model, dataset, batch_size, embedding=False):
        device = next(model.parameters()).device
        model.eval()
        chunks = []
        identities = []
        start = time.perf_counter()
        with torch.inference_mode():
            for values, _, ids in DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0):
                values = values.to(device)
                result = model.encode(values) if embedding else torch.sigmoid(model(values))
                chunks.append(result.cpu().numpy())
                identities.extend(ids)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        if identities != dataset.frame.scan_id.tolist():
            raise ValueError('Prediction order/IDs mismatch')
        return (np.concatenate(chunks), (time.perf_counter() - start) * 1000 / len(dataset))


### Boucle complète d’apprentissage, validation et reprise

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def training_signature(context, name, input_signature, device):
        provenance = context['provenance']
        data_binding = {key: provenance[key] for key in ('version', 'split_sha256', 'preprocess_sha256', 'code')}
        data_binding['extraction_version'] = provenance['extraction']['version']
        return digest(dict(preparation=data_binding, model=name, settings=asdict(context['settings']), code=source_signature(), device=str(device), input_signature=input_signature))


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def fit_model(context, name, train_dataset, validation_dataset, input_signature=''):
        s = context['settings']
        frame = context['frame']
        folder = context['run'] / name
        folder.mkdir(parents=True, exist_ok=True)
        expected_role = 'train_fusion' if name == 'M2' else 'train_encoder'
        if set(train_dataset.frame.split) != {expected_role} or set(validation_dataset.frame.split) != {'validation'}:
            raise ValueError('Invalid fitting or model-selection role')
        if set(train_dataset.frame.scan_id) & set(validation_dataset.frame.scan_id):
            raise ValueError('Training overlap')
        for dataset, role in [(train_dataset, expected_role), (validation_dataset, 'validation')]:
            reference = frame.loc[frame.split.eq(role)].sort_values('scan_id')
            actual = dataset.frame.sort_values('scan_id')
            if not np.array_equal(actual.scan_id, reference.scan_id) or not np.array_equal(actual.label_id, reference.label_id):
                raise ValueError('Dataset IDs/labels do not match the manifest')
        seed_everything(s.seed, s.threads)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = architecture(name, context['preprocess']).to(device)
        binding = training_signature(context, name, input_signature, device)
        config = dict(signature=binding, model=name, device=str(device), parameters=sum((p.numel() for p in model.parameters())), train_role=expected_role, selection_role='validation', seed=s.seed, test_used=False)
        if (folder / 'config.json').exists() and load_json(folder / 'config.json') != config:
            raise ValueError('Incompatible model cache')
        save_json(folder / 'config.json', config)
        if (folder / 'complete.json').exists():
            complete = load_json(folder / 'complete.json')
            if complete['signature'] != binding or file_hash(folder / 'best.pt') != complete['sha256']:
                raise ValueError('Checkpoint provenance mismatch')
            state = torch.load(folder / 'best.pt', map_location=device, weights_only=True)
            if state['signature'] != binding:
                raise ValueError('Embedded checkpoint signature mismatch')
            model.load_state_dict(state['weights'])
            return (model, complete)
        labels = train_dataset.frame.label_id.to_numpy()
        positive = float(labels.sum())
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(len(labels) - positive) / positive], device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=s.learning_rate, weight_decay=0.0001)
        generator = torch.Generator().manual_seed(s.seed)
        loader = DataLoader(train_dataset, batch_size=s.batch_size, shuffle=True, generator=generator, num_workers=0)
        history = []
        start_epoch = 1
        best = -1.0
        wait = 0
        if (folder / 'last.pt').exists():
            state = torch.load(folder / 'last.pt', map_location=device, weights_only=True)
            if state['signature'] != binding:
                raise ValueError('Cannot resume another model signature')
            model.load_state_dict(state['weights'])
            optimizer.load_state_dict(state['optimizer'])
            generator.set_state(state['loader_rng'].cpu())
            torch.set_rng_state(state['cpu_rng'].cpu())
            if device.type == 'cuda' and state['cuda_rng'] is not None:
                torch.cuda.set_rng_state_all(state['cuda_rng'])
            history = state['history']
            best = state['best']
            wait = state['wait']
            start_epoch = state['epoch'] + 1
        for epoch in range(start_epoch, s.epochs + 1):
            if wait >= s.patience:
                break
            model.train()
            begin = time.perf_counter()
            total = 0.0
            for batch, (values, target, _) in enumerate(loader, 1):
                optimizer.zero_grad()
                target = target.to(device)
                loss = loss_fn(model(values.to(device)), target)
                if not torch.isfinite(loss):
                    raise ValueError('Nonfinite loss')
                loss.backward()
                optimizer.step()
                total += float(loss.item()) * len(target)
                if batch % 100 == 0:
                    print(f'{name} epoch {epoch} batch {batch}/{len(loader)}', flush=True)
            scores, _ = inference(model, validation_dataset, s.batch_size)
            ap = float(average_precision_score(validation_dataset.frame.label_id, scores))
            history.append(dict(epoch=epoch, loss=total / len(labels), validation_AP=ap, seconds=time.perf_counter() - begin))
            if ap > best:
                best = ap
                wait = 0
                save_checkpoint(folder / 'best.pt', dict(weights=model.state_dict(), epoch=epoch, AP=ap, signature=binding))
            else:
                wait += 1
            save_checkpoint(folder / 'last.pt', dict(weights=model.state_dict(), optimizer=optimizer.state_dict(), epoch=epoch, best=best, wait=wait, history=history, signature=binding, loader_rng=generator.get_state(), cpu_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if device.type == 'cuda' else None))
            pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
            print(f"{name} {epoch}/{s.epochs} | loss={history[-1]['loss']:.4f} | AP validation={ap:.5f}", flush=True)
        best_state = torch.load(folder / 'best.pt', map_location=device, weights_only=True)
        model.load_state_dict(best_state['weights'])
        if best_state['signature'] != binding:
            raise ValueError('Best checkpoint belongs to another experiment')
        complete = dict(status='TRAINED', signature=binding, model=name, mode=s.mode, best_epoch=best_state['epoch'], validation_AP=best_state['AP'], train_seconds=sum((row['seconds'] for row in history)), sha256=file_hash(folder / 'best.pt'), parameters=config['parameters'], device=str(device), test_used=False)
        save_json(folder / 'complete.json', complete)
        return (model, complete)


### Entraîner une branche sur les rôles autorisés

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def fit_unimodal(context, name):
        if name not in ('U1', 'U2', 'U3', 'U4'):
            raise ValueError('Only four unimodal branches')
        frame = context['frame']
        p = context['preprocess']
        s = context['settings']
        train = ModalDataset(frame.loc[frame.split.eq('train_encoder')], p, s, name)
        validation = ModalDataset(frame.loc[frame.split.eq('validation')], p, s, name)
        return fit_model(context, name, train, validation)


### Rechercher exactement les seuils sur calibration

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def calibrate(labels, probabilities, target=0.85):
        y = np.asarray(labels)
        p = np.asarray(probabilities, float)
        if set(y) != {0, 1} or not np.isfinite(p).all():
            raise ValueError('Calibration needs two classes and finite scores')
        precision, recall, thresholds = precision_recall_curve(y, p)
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-15)
        best = np.flatnonzero(np.isclose(f1, f1.max(), atol=1e-14, rtol=0))[-1]
        candidates = np.flatnonzero(recall[:-1] >= target)
        return dict(primary=float(thresholds[best]), recall85=float(thresholds[candidates[-1]]) if len(candidates) else None)


### Calculer les métriques et la matrice de confusion

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def classification(labels, probabilities, threshold):
        y = np.asarray(labels)
        p = np.asarray(probabilities, float)
        if len(y) != len(p) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError('Invalid prediction probabilities')
        tn, fp, fn, tp = map(int, confusion_matrix(y, p >= threshold, labels=[0, 1]).ravel())
        return dict(n=len(y), prevalence=float(y.mean()), AP=float(average_precision_score(y, p)) if y.sum() else None, ROC_AUC=float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None, accuracy=(tp + tn) / len(y), precision=tp / (tp + fp) if tp + fp else 0, recall=tp / (tp + fn) if tp + fn else 0, F1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0, FPR=fp / (tn + fp) if tn + fp else None, FNR=fn / (tp + fn) if tp + fn else None, TN=tn, FP=fp, FN=fn, TP=tp, threshold=float(threshold), brier=float(brier_score_loss(y, p)))


### Vérifier la comparaison sur les mêmes scans

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def validate_alignment(ledger, reference, models):
        reference = reference.sort_values('scan_id')
        for name in models:
            selected = ledger.loc[ledger.model.eq(name)].sort_values('scan_id')
            if not selected.scan_id.is_unique or not np.array_equal(selected.scan_id, reference.scan_id) or (not np.array_equal(selected.label_id, reference.label_id)):
                raise ValueError('Unaligned multimodal predictions: ' + name)


### Bootstrap apparié par groupes

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def paired_uncertainty(ledger, reference, repeats, seed):
        reference = reference.sort_values('scan_id')
        wide = ledger.pivot(index='scan_id', columns='model', values='probability').loc[reference.scan_id]
        groups = reference.group_id.to_numpy()
        y = reference.label_id.to_numpy()
        unique = np.unique(groups)
        indices = {g: np.flatnonzero(groups == g) for g in unique}
        rng = np.random.default_rng(seed)
        records = []
        for fusion in ('M1', 'M2'):
            for baseline in ('U1', 'U2', 'U3', 'U4'):
                values = []
                for _ in range(repeats):
                    sample = np.concatenate([indices[g] for g in rng.choice(unique, len(unique), replace=True)])
                    if len(np.unique(y[sample])) < 2:
                        continue
                    values.append(average_precision_score(y[sample], wide[fusion].to_numpy()[sample]) - average_precision_score(y[sample], wide[baseline].to_numpy()[sample]))
                records.append(dict(model=fusion, reference=baseline, AP_difference=float(average_precision_score(y, wide[fusion]) - average_precision_score(y, wide[baseline])), lower95=float(np.quantile(values, 0.025)) if values else None, upper95=float(np.quantile(values, 0.975)) if values else None, bootstrap_unit='connected_site_content_group', valid_replicates=len(values)))
        return pd.DataFrame(records)


### M1 : moyenne masquée des probabilités

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def late_fusion(probabilities, mask):
        p = np.asarray(probabilities, float)
        mask = np.asarray(mask, float)
        if p.shape != mask.shape or p.shape[1] != 4:
            raise ValueError('Four aligned probabilities and masks required')
        if not np.isfinite(p).all() or (mask.sum(axis=1) == 0).any():
            raise ValueError('No observed modality or invalid score')
        return (p * mask).sum(axis=1) / mask.sum(axis=1)


### Extraire les embeddings des quatre encodeurs gelés

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def representations(context, models, frame):
        p = context['preprocess']
        s = context['settings']
        mask = availability(frame)
        embeddings = []
        probabilities = []
        timings = []
        for i, name in enumerate(('U1', 'U2', 'U3', 'U4')):
            dataset = ModalDataset(frame, p, s, name)
            z, embedding_ms = inference(models[name], dataset, s.batch_size, True)
            score, prediction_ms = inference(models[name], dataset, s.batch_size)
            embeddings.append(z * mask[:, i:i + 1])
            probabilities.append(score)
            timings.append(dict(model=name, embedding_ms=embedding_ms, prediction_ms=prediction_ms))
        return (np.concatenate(embeddings + [mask], axis=1), np.column_stack(probabilities), timings)


### M2 : apprendre sur train_fusion distinct

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def fit_fusion(context, models):
        frame = context['frame']
        run = context['run']
        p = context['preprocess']
        s = context['settings']
        parts = {role: frame.loc[frame.split.eq(role)].reset_index(drop=True) for role in ('train_fusion', 'validation')}
        encoder_ids = set(frame.loc[frame.split.eq('train_encoder')].scan_id)
        if encoder_ids & set(parts['train_fusion'].scan_id):
            raise ValueError('Fusion fitting must be held out from encoders')
        z = {role: representations(context, models, part)[0] for role, part in parts.items()}
        mean = z['train_fusion'].mean(axis=0)
        scale = z['train_fusion'].std(axis=0)
        scale[scale < 1e-06] = 1
        mean[-4:] = 0
        scale[-4:] = 1
        scaling = dict(mean=mean.tolist(), scale=scale.tolist(), fitted_ids=parts['train_fusion'].scan_id.tolist(), encoder_sha256={name: file_hash(run / name / 'best.pt') for name in ('U1', 'U2', 'U3', 'U4')})
        path = run / 'fusion_scaling.json'
        if path.exists() and load_json(path) != scaling:
            raise ValueError('Fusion inputs changed')
        save_json(path, scaling)
        ds = {role: ModalDataset(parts[role], p, s, 'M2', ((array - mean) / scale).astype(np.float32)) for role, array in z.items()}
        model, info = fit_model(context, 'M2', ds['train_fusion'], ds['validation'], digest(scaling))
        save_json(run / 'M1.json', dict(model='M1', method='equal_available_probability_mean', train_seconds=0, learned_weights=False, modalities=['url', 'html', 'image', 'technical']))
        return (model, info)


### Produire les six scores et les coûts d’inférence

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def all_scores(context, models, frame):
        z, probability, timings = representations(context, models, frame)
        late = late_fusion(probability, availability(frame))
        scaling = load_json(context['run'] / 'fusion_scaling.json')
        normalized = ((z - np.asarray(scaling['mean'], np.float32)) / np.asarray(scaling['scale'], np.float32)).astype(np.float32)
        mid, head_ms = inference(models['M2'], ModalDataset(frame, context['preprocess'], context['settings'], 'M2', normalized), context['settings'].batch_size)
        scores = {name: probability[:, i] for i, name in enumerate(('U1', 'U2', 'U3', 'U4'))}
        scores.update(M1=late, M2=mid)
        costs = {row['model']: row['prediction_ms'] for row in timings}
        costs['M1'] = sum((row['prediction_ms'] for row in timings))
        costs['M2'] = sum((row['embedding_ms'] for row in timings)) + head_ms
        return (scores, costs)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    models = {}
    display(pd.DataFrame([['U1', 'CNN URL', 'caractères → embedding → convolutions → pooling'], ['U2', 'CNN HTML', 'texte visible → tokens → convolutions → pooling'], ['U3', 'CNN Image', 'RGB → convolutions 2D → pooling'], ['U4', 'MLP technique', 'mesures numériques + masques → couches denses'], ['M1', 'Fusion tardive', 'moyenne des probabilités disponibles'], ['M2', 'Fusion intermédiaire', 'quatre embeddings gelés + masques → MLP']], columns=['ID', 'modèle', 'principe']))


### U1 — CNN URL

Objectif : apprendre les motifs de l’URL cible, avec vocabulaire train-only, PAD et UNK. Les indices d’hébergement et de protocole restent une limite d’interprétation, pas une raison d’ajuster après lecture du test.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    models['U1'], url_result = fit_unimodal(context, 'U1')
    display(pd.Series(url_result))


### U2 — CNN HTML

Objectif : apprendre le texte visible du HTML nettoyé. La longueur est fixée avant fitting, les troncatures sont enregistrées et le vocabulaire ne voit ni calibration ni test.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    models['U2'], html_result = fit_unimodal(context, 'U2')
    display(pd.Series(html_result))


### U3 — CNN Image

Objectif : apprendre des motifs visuels sans téléchargement de poids. Les états illisible/absent/uniforme sont conservés ; aucune image n’est inventée ni étiquette modifiée.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    models['U3'], image_result = fit_unimodal(context, 'U3')
    display(pd.Series(image_result))


### U4 — MLP technique

Objectif : apprendre les mesures autorisées des JSON et leurs masques. Les dates WHOIS ne sont pas un âge de page phishing ; ici elles ne sont que des indicateurs de présence de champs. Le statut HTTP n’est pas une feature.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    models['U4'], technical_result = fit_unimodal(context, 'U4')
    display(pd.Series(technical_result))


### M1 et M2 — les deux multimodaux

M1 moyenne les probabilités présentes à poids égaux renormalisés, sans poids appris. M2 concatène les embeddings masqués des encodeurs gelés, puis apprend un petit MLP sur **train_fusion**, disjoint du train des encodeurs. Son scaler voit seulement train_fusion. Les six modèles partagent validation, calibration et test. Aucune ablation supplémentaire n’est ajoutée au parcours principal.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    models['M2'], fusion_result = fit_fusion(context, models)
    display(pd.Series(fusion_result))
    print('M1 : moyenne fixe ; M2 : tête apprise sur embeddings hors train des encodeurs.')


## 5. Evaluation

Les seuils sont calculés exactement sur les scores distincts de calibration. Les checkpoints viennent de validation. Les artefacts sont gelés avant l’évaluation du test commun. AP est average precision, pas une aire trapézoïdale. Aucun calibrateur probabiliste appris n’est sous-entendu.

### Recharger les poids de cette version uniquement

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def load_bundle(context):
        models = {}
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        for name in ('U1', 'U2', 'U3', 'U4', 'M2'):
            complete = load_json(context['run'] / name / 'complete.json')
            path = context['run'] / name / 'best.pt'
            if file_hash(path) != complete['sha256']:
                raise ValueError('Model file changed')
            config = load_json(context['run'] / name / 'config.json')
            input_signature = digest(load_json(context['run'] / 'fusion_scaling.json')) if name == 'M2' else ''
            expected = training_signature(context, name, input_signature, config['device'])
            state = torch.load(path, map_location=device, weights_only=True)
            if state['signature'] != expected or complete['signature'] != expected or config['signature'] != expected:
                raise ValueError('Checkpoint/data/config signatures disagree')
            model = architecture(name, context['preprocess']).to(device)
            model.load_state_dict(state['weights'])
            model.eval()
            models[name] = model
        return models


### Geler modèles, prétraitements et seuils avant le test

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def freeze(context, models):
        folder = context['run']
        calibration = context['frame'].loc[context['frame'].split.eq('calibration')].reset_index(drop=True)
        scores, _ = all_scores(context, models, calibration)
        policies = {name: calibrate(calibration.label_id, probability) for name, probability in scores.items()}
        paths = [folder / 'preprocess.json', folder / 'fusion_scaling.json', folder / 'M1.json'] + [folder / name / 'best.pt' for name in ('U1', 'U2', 'U3', 'U4', 'M2')]
        frozen = dict(version=context['provenance']['version'], code=source_signature(), mode=context['settings'].mode, policies=policies, artifacts={p.relative_to(folder).as_posix(): file_hash(p) for p in paths}, split_sha256=context['provenance']['split_sha256'], test_used_for_selection=False, probability_calibrator='none; calibration role used for thresholds only')
        path = folder / 'frozen.json'
        if path.exists() and load_json(path) != frozen:
            raise ValueError('Frozen model/threshold changed')
        save_json(path, frozen)
        return frozen


### Contrôler les signatures avant utilisation

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def verify_frozen(context):
        run = context['run']
        frozen = load_json(run / 'frozen.json')
        if frozen['code'] != source_signature() or frozen['version'] != context['provenance']['version']:
            raise ValueError('Different data/code protocol')
        for name, expected in frozen['artifacts'].items():
            if file_hash(run / name) != expected:
                raise ValueError('Modified frozen artifact: ' + name)
        return frozen


### Évaluer les six modèles sur un registre commun

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def evaluate(context, models):
        run = context['run']
        frozen = verify_frozen(context)
        if (run / 'evaluation.json').exists():
            saved = load_json(run / 'evaluation.json')
            for name, expected in saved['files'].items():
                if file_hash(run / name) != expected:
                    raise ValueError('Evaluation ledger modified')
            return pd.read_csv(run / 'comparison.csv')
        predictions = []
        metrics = []
        for role in ('train_encoder', 'validation', 'test'):
            part = context['frame'].loc[context['frame'].split.eq(role)].reset_index(drop=True)
            scores, timings = all_scores(context, models, part)
            for name, probability in scores.items():
                predictions.append(pd.DataFrame(dict(scan_id=part.scan_id, label_id=part.label_id, group_id=part.group_id, model=name, probability=probability, split=role, mode=context['settings'].mode)))
                train_seconds = 0 if name == 'M1' else load_json(run / name / 'complete.json')['train_seconds']
                if name in ('M1', 'M2'):
                    train_seconds += sum((load_json(run / n / 'complete.json')['train_seconds'] for n in ('U1', 'U2', 'U3', 'U4')))
                for policy, threshold in frozen['policies'][name].items():
                    if threshold is not None:
                        metrics.append(dict(model=name, split=role, policy=policy, mode=context['settings'].mode, train_seconds=train_seconds, inference_ms_per_scan=timings[name], **classification(part.label_id, probability, threshold)))
        ledger = pd.concat(predictions, ignore_index=True)
        scores = pd.DataFrame(metrics)
        test = context['frame'].loc[context['frame'].split.eq('test')]
        test_predictions = ledger.loc[ledger.split.eq('test')]
        validate_alignment(test_predictions, test, MODEL_IDS)
        comparison = scores.loc[scores.split.eq('test') & scores.policy.eq('primary')].set_index('model').loc[list(MODEL_IDS)].reset_index()
        comparison['status'] = 'EVALUATED'
        ledger.to_csv(run / 'predictions.csv', index=False)
        scores.to_csv(run / 'all_metrics.csv', index=False)
        comparison.to_csv(run / 'comparison.csv', index=False)
        paired_uncertainty(test_predictions, test, context['settings'].bootstrap_replicates, context['settings'].seed).to_csv(run / 'AP_intervals.csv', index=False)
        errors = test_predictions.merge(test[['scan_id', 'provider', 'http_status', 'scan_created_at', 'html_state', 'image_state']], on='scan_id', validate='many_to_one')
        errors['threshold'] = errors.model.map({name: policy['primary'] for name, policy in frozen['policies'].items()})
        errors['error'] = np.where(errors.label_id.eq(0) & errors.probability.ge(errors.threshold), 'FP', np.where(errors.label_id.eq(1) & errors.probability.lt(errors.threshold), 'FN', 'correct'))
        errors.to_csv(run / 'error_review.csv', index=False)
        files = ('predictions.csv', 'all_metrics.csv', 'comparison.csv', 'AP_intervals.csv', 'error_review.csv')
        save_json(run / 'evaluation.json', dict(status='EVALUATED', mode=context['settings'].mode, files={name: file_hash(run / name) for name in files}, external_independent_test='NOT_AVAILABLE', labels_changed=False))
        return comparison


### Générer PR, confusion et apprentissage

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def figures(context):
        verify_frozen(context)
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        from sklearn.metrics import precision_recall_curve
        folder = context['run'] / 'figures'
        folder.mkdir(exist_ok=True)
        table = pd.read_csv(context['run'] / 'comparison.csv')
        ledger = pd.read_csv(context['run'] / 'predictions.csv')
        fig, ax = plt.subplots(figsize=(7, 4))
        for name, part in ledger.loc[ledger.split.eq('test')].groupby('model'):
            p, r, _ = precision_recall_curve(part.label_id, part.probability)
            ax.plot(r, p, label=name)
        ax.set(xlabel='Rappel phishing', ylabel='Précision', title='Test commun — ' + context['settings'].mode)
        ax.legend()
        fig.tight_layout()
        fig.savefig(folder / 'precision_recall.png', dpi=150)
        plt.close(fig)
        fig, axes = plt.subplots(2, 3, figsize=(9, 5))
        for ax, row in zip(axes.flat, table.itertuples()):
            matrix = np.array([[row.TN, row.FP], [row.FN, row.TP]])
            ax.imshow(matrix, cmap='Blues')
            for (i, j), value in np.ndenumerate(matrix):
                ax.text(j, i, str(value), ha='center', va='center')
            ax.set(title=row.model, xlabel='Prédit', ylabel='Réel', xticks=[0, 1], yticks=[0, 1])
        fig.tight_layout()
        fig.savefig(folder / 'confusion.png', dpi=150)
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(7, 4))
        for name in ('U1', 'U2', 'U3', 'U4', 'M2'):
            history = pd.read_csv(context['run'] / name / 'history.csv')
            ax.plot(history.epoch, history.validation_AP, label=name)
        ax.set(xlabel='Époque', ylabel='AP validation', title='Sélection des checkpoints')
        ax.legend()
        fig.tight_layout()
        fig.savefig(folder / 'learning.png', dpi=150)
        plt.close(fig)
        return folder


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    frozen = freeze(context, models)
    display(pd.DataFrame(frozen['policies']).T)
    comparison = evaluate(context, models)
    display(comparison)
    display(pd.read_csv(context['run'] / 'AP_intervals.csv'))


### Résultats et erreurs

Comparer AP et compromis FPR/rappel ; un score élevé ne démontre pas seul une généralisation. Les intervalles sont appariés par groupe, avec incertitude forte sur le petit lot. Les FP/FN sont des erreurs de réputation par rapport au CSV, pas une preuve que la capture montre une attaque active.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    figure_dir = figures(context)
    for name in ['precision_recall.png', 'confusion.png', 'learning.png']:
        display(NotebookImage(filename=str(figure_dir / name)))
    errors = pd.read_csv(context['run'] / 'error_review.csv')
    display(errors.groupby(['model', 'provider', 'error']).size().rename('n').to_frame())


### Expériences prédéfinies : toutes les combinaisons de modalités (RQ1)

On évalue **6 paires, 4 triplets et la combinaison à quatre modalités** par deux méthodes :
1. **Fusion tardive** : moyenne non pondérée des probabilités disponibles parmi les modalités demandées, sans entraînement supplémentaire.
2. **Fusion de représentations linéaire** : régression logistique sur les embeddings gelés des unimodaux, entraînée uniquement sur `train_fusion` (un diagnostic complémentaire à M2, qui reste une fusion MLP).

Les combinaisons sont énumérées **avant toute lecture du test** ; les transformations de la tête linéaire sont ajustées sur `train_fusion` et les seuils sur `calibration`. Si aucune des modalités demandées n'est présente, le système **s'abstient** au lieu d'inventer une prédiction. Chaque combinaison produit une mesure de couverture sur le test complet et une évaluation séparée sur **la même cohorte des quatre modalités présentes**. Aucun gagnant n'est déclaré sur le test déjà exploré ; les tableaux constituent une comparaison interne exploratoire.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    from itertools import combinations
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    import joblib

    COMBO_MODALITIES = ('URL', 'HTML', 'IMAGE', 'TECH')
    COMBO_SPECS = tuple(choice for k in (2, 3, 4) for choice in combinations(range(4), k))
    assert len(COMBO_SPECS) == 11

    def combo_role_arrays(context, models, role, binding):
        folder = context['run'] / 'combinations'
        part = context['frame'].loc[context['frame'].split.eq(role)].reset_index(drop=True)
        path = folder / (role + '.npz')
        meta = folder / (role + '.json')
        if meta.exists():
            info = load_json(meta)
            if info['binding'] != binding or file_hash(path) != info['sha256']:
                raise ValueError('Combination representation cache changed: ' + role)
            with np.load(path, allow_pickle=False) as saved:
                if saved['ids'].tolist() != part.scan_id.tolist():
                    raise ValueError('Combination IDs differ from manifest')
                z, probability, mask = (saved[name].copy() for name in ('z', 'probability', 'mask'))
        else:
            z, probability, _ = representations(context, models, part)
            mask = availability(part)
            temporary = folder / (role + '.tmp.npz')
            np.savez_compressed(temporary, z=z, probability=probability, mask=mask,
                                ids=part.scan_id.to_numpy(dtype=str))
            temporary.replace(path)
            save_json(meta, dict(binding=binding, sha256=file_hash(path)))
        if z.shape != (len(part), 132) or probability.shape != (len(part), 4) or mask.shape != (len(part), 4):
            raise ValueError('Expected four aligned 32-dimensional embeddings and four masks')
        if not all(np.isfinite(value).all() for value in (z, probability, mask)):
            raise ValueError('Nonfinite combination representations')
        return part, z, probability, mask

    def combo_selected(z, probability, mask, choice):
        selected = np.asarray(choice, dtype=int)
        x = np.concatenate([z[:, 32*i:32*(i+1)] for i in choice] + [mask[:, selected]], axis=1)
        eligible = mask[:, selected].any(axis=1)
        weights = mask[:, selected]
        p = np.full(len(z), np.nan)
        p[eligible] = ((probability[:, selected] * weights).sum(axis=1)[eligible]
                       / weights.sum(axis=1)[eligible])
        return x, p, eligible

    def combo_cohort_metrics(labels, probability, threshold, selected):
        selected = np.asarray(selected, bool) & np.isfinite(probability)
        if not selected.any() or threshold is None:
            keys = ('prevalence', 'AP', 'ROC_AUC', 'accuracy', 'precision', 'recall', 'F1',
                    'FPR', 'FNR', 'TN', 'FP', 'FN', 'TP', 'threshold', 'brier')
            return dict(n=0, status='NOT_FEASIBLE', **dict.fromkeys(keys))
        result = classification(np.asarray(labels)[selected], probability[selected], threshold)
        result['status'] = 'EVALUATED' if np.unique(np.asarray(labels)[selected]).size == 2 else 'ONE_CLASS_ONLY'
        return result

    def run_combo_study(context, models):
        folder = context['run'] / 'combinations'
        folder.mkdir(exist_ok=True, parents=True)
        central_frozen = verify_frozen(context)
        binding = digest(dict(code=source_signature(), version=context['provenance']['version'],
                              central_frozen=central_frozen, specs=COMBO_SPECS,
                              linear=dict(C=1.0, max_iter=500, class_weight='balanced', seed=context['settings'].seed)))
        frozen_path = folder / 'frozen.json'
        if frozen_path.exists():
            frozen = load_json(frozen_path)
            if frozen['binding'] != binding:
                raise ValueError('Combination configuration changed; start a distinct experiment')
            for name, expected in frozen['artifacts'].items():
                if file_hash(folder / name) != expected:
                    raise ValueError('Modified combination checkpoint or calibration: ' + name)
        else:
            # No test representations or test scores are read before every head and threshold is frozen.
            train, tz, tp, tm = combo_role_arrays(context, models, 'train_fusion', binding)
            calibration, cz, cp, cm = combo_role_arrays(context, models, 'calibration', binding)
            policies, artifacts = [], {}
            for choice in COMBO_SPECS:
                label = '+'.join(COMBO_MODALITIES[i] for i in choice)
                tx, _, train_ok = combo_selected(tz, tp, tm, choice)
                cx, late_cp, cal_ok = combo_selected(cz, cp, cm, choice)
                for method in ('LATE', 'LINEAR_EMBED'):
                    name = method + '__' + label
                    checkpoint = 'LINEAR_EMBED__' + label.replace('+', '_') + '.joblib'
                    reason, n_fit = None, 0
                    if method == 'LINEAR_EMBED':
                        if np.unique(train.label_id.to_numpy()[train_ok]).size != 2:
                            reason = 'train_fusion lacks both classes among available inputs'
                            scores = np.full(len(calibration), np.nan)
                        else:
                            model = make_pipeline(StandardScaler(), LogisticRegression(
                                C=1.0, max_iter=500, class_weight='balanced', random_state=context['settings'].seed))
                            model.fit(tx[train_ok], train.label_id.to_numpy()[train_ok])
                            temporary = folder / (checkpoint + '.tmp')
                            joblib.dump(model, temporary)
                            temporary.replace(folder / checkpoint)
                            artifacts[checkpoint] = file_hash(folder / checkpoint)
                            scores = np.full(len(calibration), np.nan)
                            if cal_ok.any():
                                scores[cal_ok] = model.predict_proba(cx[cal_ok])[:, 1]
                            n_fit = int(train_ok.sum())
                    else:
                        scores = late_cp
                    valid = cal_ok & np.isfinite(scores)
                    if np.unique(calibration.label_id.to_numpy()[valid]).size != 2:
                        reason = reason or 'calibration lacks both classes among available inputs'
                    policy = calibrate(calibration.label_id.to_numpy()[valid], scores[valid]) if reason is None else {}
                    policies.append(dict(model=name, method=method, modalities=label, choice=list(choice),
                        threshold=policy.get('primary'), threshold_recall85=policy.get('recall85'),
                        n=int(valid.sum()), train_fusion_rows=n_fit,
                        status='FROZEN' if reason is None else 'NOT_FEASIBLE', reason=reason,
                        checkpoint=checkpoint if method == 'LINEAR_EMBED' and n_fit else None))
                    print('Combination frozen:', name, policies[-1]['status'], flush=True)
            pd.DataFrame(policies).to_csv(folder / 'calibration_policies.csv', index=False)
            artifacts['calibration_policies.csv'] = file_hash(folder / 'calibration_policies.csv')
            for role in ('train_fusion', 'calibration'):
                for suffix in ('.npz', '.json'):
                    name = role + suffix
                    artifacts[name] = file_hash(folder / name)
            frozen = dict(binding=binding, policies=policies, artifacts=artifacts,
                          selection_on_test=False, train_role='train_fusion', threshold_role='calibration')
            save_json(frozen_path, frozen)
        # All 22 policies are durable and immutable before this point.
        test, z, probability, mask = combo_role_arrays(context, models, 'test', binding)
        complete = mask.all(axis=1)
        records, predictions = [], []
        for item in frozen['policies']:
            x, scores, eligible = combo_selected(z, probability, mask, item['choice'])
            if item['status'] != 'FROZEN':
                scores = np.full(len(test), np.nan)
                eligible = np.zeros(len(test), bool)
            elif item['method'] == 'LINEAR_EMBED':
                model = joblib.load(folder / item['checkpoint'])
                scores = np.full(len(test), np.nan)
                if eligible.any():
                    scores[eligible] = model.predict_proba(x[eligible])[:, 1]
            for cohort, selector in (('available', eligible), ('all_four_available', complete & eligible)):
                records.append(dict(model=item['model'], method=item['method'], modalities=item['modalities'],
                    cohort=cohort, test_full_n=len(test), coverage=float(selector.mean()), reason=item['reason'],
                    **combo_cohort_metrics(test.label_id.to_numpy(), scores, item['threshold'], selector)))
            predictions.extend(dict(scan_id=scan, model=item['model'],
                probability=float(p) if np.isfinite(p) else None, label_id=int(y), eligible=bool(ok),
                complete_case=bool(cc), threshold=item['threshold'], abstained=not bool(ok))
                for scan, p, y, ok, cc in zip(test.scan_id, scores, test.label_id, eligible, complete))
        result = pd.DataFrame(records)
        result.to_csv(folder / 'metrics.csv', index=False)
        pd.DataFrame(predictions).to_csv(folder / 'predictions.csv', index=False)
        # Use the existing common-test registry: no second inference of the six central models.
        ledger = pd.read_csv(context['run'] / 'predictions.csv', dtype={'scan_id': str})
        base = []
        for name in MODEL_IDS:
            chosen = ledger.loc[ledger.model.eq(name) & ledger.split.eq('test')].set_index('scan_id')
            if not chosen.index.is_unique or set(chosen.index) != set(test.scan_id):
                raise ValueError('Central predictions do not match common test IDs')
            chosen = chosen.loc[test.scan_id]
            if not np.array_equal(chosen.label_id.to_numpy(), test.label_id.to_numpy()):
                raise ValueError('Central prediction labels differ from manifest')
            scores = chosen.probability.to_numpy()
            base.append(dict(model=name, method='BASELINE_6', cohort='all_four_available',
                test_full_n=len(test), coverage=float(complete.mean()),
                **combo_cohort_metrics(test.label_id.to_numpy(), scores, central_frozen['policies'][name]['primary'], complete)))
        base = pd.DataFrame(base)
        base.to_csv(folder / 'baseline_complete_case.csv', index=False)
        all_28 = pd.concat([base, result.loc[result.cohort.eq('all_four_available')]], ignore_index=True)
        assert len(all_28) == 28 and all_28.model.nunique() == 28
        comparable = all_28.loc[all_28.status.eq('EVALUATED')]
        if len(comparable) and comparable.n.nunique() != 1:
            raise ValueError('Models were evaluated on different complete-case cohorts')
        all_28.to_csv(folder / 'all_28_comparable_models.csv', index=False)
        files = ['metrics.csv', 'predictions.csv', 'baseline_complete_case.csv', 'all_28_comparable_models.csv']
        save_json(folder / 'study.json', dict(status='EVALUATED', binding=binding,
            combination_count=11, additional_models=22, central_models=6,
            complete_case_two_classes=np.unique(test.label_id.to_numpy()[complete]).size == 2,
            test_full_n=len(test), all_four_complete_n=int(complete.sum()),
            independent_external_test=False, selection_on_test=False,
            incomplete_inputs='use available requested modalities; abstain if none',
            files={name: file_hash(folder / name) for name in files}))
        return result, base

    combo_results, complete_baselines = run_combo_study(context, models)
    all_28 = pd.read_csv(context['run'] / 'combinations/all_28_comparable_models.csv')
    display(all_28[['model', 'method', 'status', 'n', 'AP', 'F1', 'recall', 'FPR', 'coverage']])
    print('28 configurations predefinies. Comparaison interne exploratoire ; aucun gagnant final choisi sur le test.')


### Sensibilité des modèles à leurs entrées (validation uniquement)

Permutation de chacune des 21 features techniques sur les observations de validation, avec modèle U4 gelé. Ces diminutions d'AP sont des **diagnostics d'influence prédictive, pas des preuves de causalité**. Le protocole HTTP/HTTPS est également testé par perturbation de l'URL du modèle U1 à checkpoint fixe ; les URL synthétiques n'ont pas de nouveau label, on rapporte donc uniquement des changements de score/décision.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def input_sensitivity_report(context, models):
        run=context['run']/'feature_audit'
        run.mkdir(exist_ok=True,parents=True)
        part=context['frame'].loc[context['frame'].split.eq('validation')].reset_index(drop=True)
        s=context['settings']
        base,_=inference(models['U4'],ModalDataset(part,context['preprocess'],s,'U4'),s.batch_size)
        base_ap=float(average_precision_score(part.label_id,base))
        rng=np.random.default_rng(s.seed)
        records=[]
        for feature in FEATURES:
            changed=part.copy()
            changed[feature]=part[feature].to_numpy()[rng.permutation(len(part))]
            pred,_=inference(models['U4'],ModalDataset(changed,context['preprocess'],s,'U4'),s.batch_size)
            records.append(dict(feature=feature,modality='technical_json',baseline_validation_AP=base_ap,
                       permuted_validation_AP=float(average_precision_score(part.label_id,pred)),
                       AP_drop=base_ap-float(average_precision_score(part.label_id,pred))))
        technical=pd.DataFrame(records).sort_values('AP_drop',ascending=False)
        technical.to_csv(run/'technical_permutation_validation.csv',index=False)
        base_url,_=inference(models['U1'],ModalDataset(part,context['preprocess'],s,'U1'),s.batch_size)
        threshold=verify_frozen(context)['policies']['U1']['primary']
        from urllib.parse import urlsplit,urlunsplit
        probes=[]
        for proto in ('http','https'):
            changed=part.copy()
            def replace_scheme(value):
                u=urlsplit(value)
                return urlunsplit((proto,u.netloc,u.path,u.query,u.fragment)) if u.netloc else value
            changed['target_url']=changed.target_url.map(replace_scheme)
            prob,_=inference(models['U1'],ModalDataset(changed,context['preprocess'],s,'U1'),s.batch_size)
            probes.append(dict(probe='force_'+proto,n=len(part),
                     mean_abs_probability_delta=float(np.mean(np.abs(prob-base_url))),
                     prediction_flip_rate=float(np.mean((prob>=threshold)!=(base_url>=threshold))),
                     synthetic_urls_no_ground_truth=True))
        pd.DataFrame(probes).to_csv(run/'url_scheme_sensitivity_validation.csv',index=False)
        return technical,pd.DataFrame(probes)

    technical_importance,url_scheme_probes = input_sensitivity_report(context,models)
    display(technical_importance)
    display(url_scheme_probes)


### RQ2 — validation chronologique nouvellement construite

Les frontières viennent des quantiles temporels bruts fixés par les fractions du protocole. Tout groupe traversant une frontière est purgé. Les mêmes six architectures sont entraînées à neuf sur ces rôles temporels si chacun contient les deux classes. Sinon le résultat est NOT_FEASIBLE, sans déplacement de dates ni recherche d’une fenêtre donnant un beau score. Le corpus ayant déjà été exploré, ce test reste interne ; aucun test externe scellé n’est disponible.

### Entraîner les mêmes six modèles pour un protocole donné

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def model_stage(context):
        models = {}
        for name in MODEL_IDS[:4]:
            models[name], _ = fit_unimodal(context, name)
        models['M2'], _ = fit_fusion(context, models)
        freeze(context, models)
        result = evaluate(context, models)
        figures(context)
        return result


### Valider chronologiquement ou déclarer la limite

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def temporal_validation(settings):
        chronological = replace(settings, protocol='chronological')
        report = ROOT / 'artifacts' / ('temporal_' + settings.mode.lower() + '.json')
        try:
            context = prepare(chronological)
        except NotFeasible as error:
            status = dict(status='NOT_FEASIBLE', reason=str(error), dates_modified=False, policy='fixed cumulative raw-date quantiles, crossing-site/content groups purged', model_selection_on_temporal_test=False)
            save_json(report, status)
            return status
        result = model_stage(context)
        status = dict(status='EVALUATED', run=context['run'].relative_to(ROOT).as_posix(), mode=settings.mode, code=source_signature(), models=result.to_dict('records'), evaluation='chronological internal diagnostic; not an independent external dataset')
        save_json(report, status)
        return status


### Répondre à RQ1, RQ2 et RQ3 selon les exécutions

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def conclusions(context, temporal=None):
        comparison = pd.read_csv(context['run'] / 'comparison.csv').set_index('model')
        mode = context['settings'].mode
        best = comparison.loc[list(MODEL_IDS[:4]), 'AP'].idxmax()
        measured = f"U1–U4 : meilleur AP observé {best} ({comparison.loc[best, 'AP']:.4f}); M1={comparison.loc['M1', 'AP']:.4f}, M2={comparison.loc['M2', 'AP']:.4f}."
        rq1 = 'SMOKE_TEST : fonctionnement uniquement, aucun gagnant scientifique démontré. ' + measured if mode == 'SMOKE_TEST' else measured + ' Interpréter avec les intervalles par groupes et le compromis FPR/rappel.'
        rq2 = 'NOT_RUN' if temporal is None else temporal['status']
        text = f"# Résultats du nouveau pipeline — {mode}\n\nRQ1 : {rq1}\n\nRQ2 : étude chronologique {rq2}. Les fenêtres sans deux classes ne sont pas artificiellement réparées. Aucun test externe indépendant n'est disponible ; le corpus a déjà été exploré. Aucun score temporel ne choisit un modèle ou un seuil.\n\nRQ3 : coûts d'extraction, d'entraînement et d'inférence enregistrés. Le benchmark équivalent pandas/Spark précise les étapes mesurées et les absences de dépendances. Spark local ne démontre aucun gain de cluster. Cache froid non contrôlé.\n\nLes nouveaux labels sont ceux du CSV original, sans correction automatique. Les erreurs de capture, familles inconnues et biais HTTPS/hébergement/source/date limitent l'interprétation. Une collecte équilibrée simultanée et un nouveau test scellé restent nécessaires.\n"
        (context['run'] / 'SUMMARY.md').write_text(text, encoding='utf-8')
        return text


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    temporal_result = temporal_validation(settings)
    display(temporal_result)


### RQ3 — benchmark équitable et reprise

Scénario A : un fichier de données Parquet non partitionné dans un répertoire pour chaque moteur. Scénario B : répertoires Hive split=role dans les deux moteurs. Même cohorte, schéma, colonnes, validation, conversion, compression et vérification exacte après réouverture, incluse dans le temps. Les tentatives interrompues sont conservées ; chaque mesure complète a un journal et des empreintes. Le démarrage Spark est séparé, le cache OS non contrôlé et aucun cluster multi-nœud n’est revendiqué.

### Définir le schéma Spark

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def spark_schema():
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
        return StructType([StructField('scan_id', StringType(), False), StructField('label_id', IntegerType(), False), StructField('split', StringType(), False)] + [StructField(name, FloatType(), True) for name in FEATURES])


### Mesurer des opérations pandas et Spark équivalentes

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def canonical_benchmark_table(frame):
        data = frame[['scan_id', 'label_id', 'split', *FEATURES]].copy()
        data['scan_id'] = data.scan_id.astype('string')
        data['split'] = data.split.astype('string')
        data['label_id'] = data.label_id.astype('int32')
        for name in FEATURES:
            data[name] = data[name].astype('float32')
        return data.sort_values('scan_id').reset_index(drop=True)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def validate_benchmark_table(frame, n, positives):
        if len(frame) != n or not frame.scan_id.is_unique or int(frame.label_id.sum()) != positives:
            raise ValueError('Benchmark validation failure')


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def write_pandas_dataset(frame, target, partitioned):
        target = Path(target)
        if target.exists():
            raise FileExistsError('Never overwrite a benchmark attempt')
        target.mkdir(parents=True)
        if partitioned:
            for role, part in frame.groupby('split', observed=True):
                if role not in ('train_encoder', 'train_fusion', 'validation', 'calibration', 'test'):
                    raise ValueError('Unsafe partition value')
                directory = target / f'split={role}'
                directory.mkdir()
                part.drop(columns='split').to_parquet(directory / 'p.parquet', index=False, compression='snappy')
        else:
            frame.to_parquet(target / 'p.parquet', index=False, compression='snappy')


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    class BenchmarkMemory:
        """Sample summed process/JVM RSS; shared pages can be counted twice."""

        def __init__(self):
            self.stop = threading.Event()
            self.peak = 0
            self.thread = None

        def sample(self):
            while not self.stop.is_set():
                process = psutil.Process()
                total = process.memory_info().rss
                for child in process.children(recursive=True):
                    try:
                        total += child.memory_info().rss
                    except psutil.Error:
                        pass
                self.peak = max(self.peak, total)
                self.stop.wait(0.05)

        def __enter__(self):
            self.thread = threading.Thread(target=self.sample, daemon=True)
            self.thread.start()
            return self

        def __exit__(self, *args):
            self.stop.set()
            self.thread.join(timeout=2)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def measure_parquet_case(context, frame, scenario, engine, repetition, fraction, spark=None):
        s = context['settings']
        expected = canonical_benchmark_table(frame)
        n = len(expected)
        positives = int(expected.label_id.sum())
        binding = dict(run=context['provenance']['version'], code=source_signature(), scenario=scenario, engine=engine, repetition=repetition, fraction=fraction, threads=s.threads, rows=digest(expected.to_json(orient='split')))
        directory = context['run'] / 'bd2' / digest(binding)[:20]
        directory.mkdir(parents=True, exist_ok=True)
        ledger = directory / 'measurement.json'
        if ledger.exists():
            completed = load_json(ledger)
            if completed['binding'] != binding:
                raise ValueError('Benchmark binding changed')
            for relative, expected_hash in completed['output_hashes'].items():
                if file_hash(directory / relative) != expected_hash:
                    raise ValueError('Benchmark output changed')
            return completed['result']
        attempt = 1
        while (directory / f'a{attempt:04d}').exists():
            attempt += 1
        attempt_dir = directory / f'a{attempt:04d}'
        attempt_dir.mkdir()
        source = attempt_dir / 'input.parquet'
        expected.to_parquet(source, index=False, compression='snappy')
        target = attempt_dir / 'o'
        partitioned = scenario == 'B_partitioned'
        pa.set_cpu_count(s.threads)
        with BenchmarkMemory() as memory:
            start = time.perf_counter()
            if engine == 'pandas_pyarrow':
                data = pd.read_parquet(source)
                validate_benchmark_table(data, n, positives)
                write_pandas_dataset(canonical_benchmark_table(data), target, partitioned)
            elif engine == 'spark_local' and spark is not None:
                from pyspark.sql import functions as F
                distributed = spark.read.schema(spark_schema()).parquet(str(source))
                stats = distributed.agg(F.count('*').alias('n'), F.countDistinct('scan_id').alias('unique'), F.sum('label_id').alias('positives')).first()
                if stats.n != n or stats.unique != n or stats.positives != positives:
                    raise ValueError('Spark validation failure')
                distributed = distributed.select(F.col('scan_id').cast('string'), F.col('label_id').cast('int').alias('label_id'), F.col('split').cast('string'), *[F.col(name).cast('float').alias(name) for name in FEATURES])
                if partitioned:
                    distributed.repartition(5, 'split').write.mode('errorifexists').option('compression', 'snappy').partitionBy('split').parquet(str(target))
                else:
                    distributed.coalesce(1).write.mode('errorifexists').option('compression', 'snappy').parquet(str(target))
            else:
                raise ValueError('Unknown/unavailable engine')
            restored = canonical_benchmark_table(pd.read_parquet(target))
            validate_benchmark_table(restored, n, positives)
            pd.testing.assert_frame_equal(restored, expected, check_exact=True)
            seconds = time.perf_counter() - start
        files = sorted(target.rglob('*.parquet'))
        if not partitioned and len(files) != 1:
            raise ValueError('Scenario A requires one data file')
        result = dict(engine=engine, scenario=scenario, status='MEASURED', operation='read_validate_cast_write_readback_verify', n=n, fraction=fraction, repetition=repetition, repetitions_planned=s.benchmark_repeats, seconds=seconds, scans_per_second=n / seconds, input_bytes=source.stat().st_size, output_bytes=sum((p.stat().st_size for p in files)), data_files=len(files), peak_process_tree_rss_bytes=memory.peak, memory_scope='sampled RSS sum; shared pages may be double-counted', logical_cpus=os.cpu_count(), threads=s.threads, cache='OS_uncontrolled', compression='snappy', mode=s.mode, master=f'local[{s.threads}]' if engine == 'spark_local' else None, multinode=False, attempt=attempt)
        save_json(ledger, dict(binding=binding, result=result, output_hashes={p.relative_to(directory).as_posix(): file_hash(p) for p in files}))
        return result


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def benchmark(context):
        report = dependency_report()
        s = context['settings']
        directory = context['run'] / 'bd2'
        directory.mkdir(exist_ok=True)
        records = []
        spark = None
        startup = None
        if report['spark_status'] == 'AVAILABLE' and report['available_ram_gb'] >= 2:
            from pyspark.sql import SparkSession
            try:
                start = time.perf_counter()
                spark = SparkSession.builder.master(f'local[{s.threads}]').appName('PhiSharkFairBenchmark').config('spark.ui.enabled', 'false').getOrCreate()
                startup = time.perf_counter() - start
            except Exception as error:
                records.append(dict(engine='spark_local', status='FAIL', reason=type(error).__name__ + ':' + str(error)[:180]))
        else:
            records.append(dict(engine='spark_local', status='NOT_AVAILABLE' if report['spark_status'] != 'AVAILABLE' else 'BLOCKED', reason='PySpark/Java absent or less than 2 GB available RAM'))
        frame = canonical_benchmark_table(context['frame'])
        try:
            for fraction in s.benchmark_fractions:
                subset = frame.iloc[:max(1, int(len(frame) * fraction))]
                for repetition in range(s.benchmark_repeats):
                    for scenario in ('A_unpartitioned', 'B_partitioned'):
                        records.append(measure_parquet_case(context, subset, scenario, 'pandas_pyarrow', repetition, fraction))
                        if spark is not None:
                            records.append(measure_parquet_case(context, subset, scenario, 'spark_local', repetition, fraction, spark))
        finally:
            if spark is not None:
                spark.stop()
        result = pd.DataFrame(records)
        result.to_csv(directory / 'measurements.csv', index=False)
        save_json(directory / 'scope.json', dict(extraction_seconds=context['provenance']['extraction']['seconds'], spark_startup_seconds=startup, spark_startup_included=False, cold_cache='NOT_CONTROLLED', multinode=False, resources=report, scenarios=dict(A_unpartitioned='one Parquet data file for both engines', B_partitioned='Hive split=role for both engines'), integrity='identical readback and exact canonical frame equality included in time'))
        return result


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    benchmark_result = benchmark(context)
    display(benchmark_result)
    display(Markdown(conclusions(context, temporal_result)))


### Diagnostic ciblé des signaux de disponibilité — validation uniquement

Une baseline utilise seulement les flags de collecte. Deux têtes linéaires comparent les embeddings avec/sans les quatre masques explicites. Elles apprennent sur les rôles autorisés et n’évaluent que validation au seuil diagnostic fixe 0,5. Elles ne sélectionnent ni le modèle principal ni son seuil, et ne suppriment pas les proxys implicites présents dans les embeddings.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def availability_ablation(context, models):
        frame = context['frame']
        run = context['run']
        s = context['settings']
        train = frame.loc[frame.split.eq('train_encoder')].reset_index(drop=True)
        fusion = frame.loc[frame.split.eq('train_fusion')].reset_index(drop=True)
        validation = frame.loc[frame.split.eq('validation')].reset_index(drop=True)

        def flags(part):
            return np.column_stack((availability(part), ~np.isfinite(part[list(FEATURES)].to_numpy(float)), part.uniform.to_numpy(), part.http_status.isna().to_numpy(), part.capture_errors.str.len().gt(0).to_numpy())).astype(float)
        records = []
        baseline = LogisticRegression(max_iter=500, random_state=s.seed).fit(flags(train), train.label_id)
        probability = baseline.predict_proba(flags(validation))[:, 1]
        records.append(dict(diagnostic='collection_flags_only', fitting_role='train_encoder', evaluation_role='validation', **classification(validation.label_id, probability, 0.5)))
        z_train, _, _ = representations(context, models, fusion)
        z_validation, _, _ = representations(context, models, validation)
        for name, size in [('embeddings_and_explicit_masks', 132), ('embeddings_without_explicit_masks', 128)]:
            scaler = StandardScaler().fit(z_train[:, :size])
            head = LogisticRegression(max_iter=500, random_state=s.seed).fit(scaler.transform(z_train[:, :size]), fusion.label_id)
            probability = head.predict_proba(scaler.transform(z_validation[:, :size]))[:, 1]
            records.append(dict(diagnostic=name, fitting_role='train_fusion', evaluation_role='validation', **classification(validation.label_id, probability, 0.5)))
        result = pd.DataFrame(records)
        result.to_csv(run / 'availability_ablation_validation.csv', index=False)
        save_json(run / 'availability_ablation_scope.json', dict(primary_models_changed=False, test_used=False, fixed_diagnostic_threshold=0.5, interpretation='removes explicit masks only; embeddings can still encode availability and collection effects', small_sample_warning=s.mode == 'SMOKE_TEST'))
        return result


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    availability_diagnostic = availability_ablation(context,models)
    display(availability_diagnostic)


## 6. Deployment — prototype local

La prédiction d’une archive vérifie les artefacts gelés puis utilise les modalités réellement observées. Avec une URL seule, seule U1 est exécutée. Aucun téléchargement de page, aucune exécution de JavaScript. Ce prototype prédit une réputation URL, pas une certitude de phishing actuellement visible.

### Prédire une URL seule sans réseau

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def predict_url(context, url):
        frozen = verify_frozen(context)
        models = load_bundle(context)
        frame = pd.DataFrame([dict(scan_id='url-only', label_id=0, target_url=str(url))])
        probability, _ = inference(models['U1'], ModalDataset(frame, context['preprocess'], context['settings'], 'U1'), 1)
        threshold = frozen['policies']['U1']['primary']
        probability = float(probability[0])
        return dict(model='U1', probability=probability, threshold=threshold, predicted_class=int(probability >= threshold), observed_modalities=['URL'], network_access=False, mode=context['settings'].mode)


### Prédire une archive avec ses modalités observées

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def predict_scan(context, path):
        frozen = verify_frozen(context)
        models = load_bundle(context)
        s = context['settings']
        scan = read_archive(path, s.image_size)
        metadata = scan['data'].get('metadata.json', {})
        target = metadata.get('target') or scan['data'].get('data/redirection.json', {}).get('original_url', '')
        temporary_root = ROOT / 'artifacts/tmp'
        temporary_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=temporary_root) as temp:
            image = Path(temp) / 'image.npy'
            np.save(image, scan['image'])
            row = dict(scan_id=Path(path).stem, label_id=0, target_url=target, html_text=scan['text'], image_relpath=image.relative_to(ROOT).as_posix(), html_state=scan['html_state'], image_state=scan['image_state'])
            row.update(zip(FEATURES, extract_numeric(scan['data'])))
            frame = pd.DataFrame([row])
            scores, _ = all_scores(context, models, frame)
            mask = availability(frame)[0]
        result = {}
        for i, name in enumerate(MODEL_IDS):
            if i < 4 and (not mask[i]):
                continue
            probability = float(scores[name][0])
            threshold = frozen['policies'][name]['primary']
            result[name] = dict(probability=probability, threshold=threshold, predicted_class=int(probability >= threshold))
        return dict(mode=s.mode, scores=result, availability=mask.tolist(), html_state=scan['html_state'], image_state=scan['image_state'], warnings=scan['errors'], meaning='URL reputation, not proof of currently active phishing', network_access=False)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    demo = context['frame'].loc[context['frame'].split.eq('test')].sort_values('scan_id').iloc[0]
    display(predict_scan(context, RAW_DIRECTORY / demo.archive_filename))
    display(predict_url(context, 'https://example.invalid/login'))
    print('Dossier de cette expérience :', context['run'])
    print('SMOKE = fonctionnalité seulement ; FULL et nouveau lot externe requis pour une conclusion finale.')


### Vérification finale de cette exécution

Le tableau central reste limité aux six modèles. Les sorties sont calculées à partir de nouveaux poids et de nouveaux registres. Les résultats du mode court démontrent le fonctionnement ; ils ne démontrent ni convergence, ni avantage multimodal scientifique. Spark absent, protocole temporel inexploitable et test externe indisponible restent explicitement signalés.

In [ ]:
if MODE == 'FULL_EXPERIMENT':
    assert comparison.model.tolist()==list(MODEL_IDS)
    assert comparison.n.nunique()==1
    check_partitions(context['frame'])
    assert context['preprocess']['fitted_ids']==context['frame'].loc[context['frame'].split.eq('train_encoder')].scan_id.tolist()
    display(comparison[['model','mode','n','AP','accuracy','precision','recall','F1','FPR','FNR']])
    print('Notebook intégral exécuté. Dossier de résultats :',context['run'])
    print('Aucun module Python du projet n’a été nécessaire à cette exécution.')


### Rapport CRISP-DM genere depuis les resultats de cette execution


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    def export_crisp_dm_report(context, temporal_result, benchmark_result):
        def table(frame):
            def value(x):
                if x is None or (isinstance(x, (float, np.floating)) and not np.isfinite(x)):
                    return '—'
                return (f'{x:.4f}' if isinstance(x, (float, np.floating)) else str(x)).replace('|', '/')
            rows = ['| ' + ' | '.join(map(str, frame.columns)) + ' |',
                    '| ' + ' | '.join(['---']*len(frame.columns)) + ' |']
            rows += ['| ' + ' | '.join(value(x) for x in row) + ' |' for row in frame.itertuples(index=False, name=None)]
            return '\n'.join(rows)
        frozen = verify_frozen(context)
        run = context['run']
        central = pd.read_csv(run/'comparison.csv')
        combos = pd.read_csv(run/'combinations/all_28_comparable_models.csv')
        counts = pd.DataFrame(context['provenance']['class_counts'])
        feature_scope = '21 features JSON ; sources, dates, labels, hebergeurs et statuts HTTP hors X technique.'
        text = [
            '# PhiShark2026 — rapport final CRISP-DM',
            '\nMode : **'+context['settings'].mode+'**. Code : `'+source_signature()+'`.',
            '\n## 1. Business Understanding',
            'Cible : reputation de l URL annotee, distincte du contenu effectivement capture. Un HTTP404 ne change pas le label.',
            'RQ1 : comparaison des unimodaux et des fusions. RQ2 : diagnostic chronologique. RQ3 : cout pandas/Spark a perimetre comparable.',
            '\n## 2. Data Understanding', feature_scope,
            f"Observations brutes : {context['provenance']['raw_rows']}. En quarantaine : {context['provenance']['quarantined_rows']}. Admissibles avant purge : {context['provenance']['eligible_rows']}.",
            'Les associations globales du CSV sont descriptives. Les audits des features JSON sont calcules sur train_encoder uniquement.',
            '\n## 3. Data Preparation', table(counts),
            'Identites site/locataire/contenu et encodages traversant les roles interdits. Vocabulaires, imputation et normalisation ajustes sur train_encoder uniquement.',
            'Les captures absentes restent declarees par masques. Aucun label ni aucune date modifie automatiquement.',
            '\n## 4. Modeling',
            'U1 CNN URL sans schema HTTP/HTTPS ; U2 CNN texte HTML ; U3 CNN image (FULL 96x96) ; U4 MLP technique. Embeddings de 32 valeurs par branche.',
            'M1 moyenne les probabilites disponibles. M2 apprend un MLP sur embeddings geles et masques, sur train_fusion.',
            '11 sous-ensembles (six paires, quatre triplets, un quadruplet) : LATE et LINEAR_EMBED, soit 22 etudes supplementaires. Aucune recherche de gagnant sur le test.',
            'Checkpoints choisis sur validation ; seuils exclusivement sur calibration. Politiques et artefacts sauvegardes avant evaluation.',
            '\n## 5. Evaluation — RQ1',
            table(central[['model','n','AP','precision','recall','F1','FPR','FNR']]),
            '\n### 28 configurations sur la cohorte commune',
            table(combos[['model','status','n','AP','precision','recall','F1','FPR','coverage']]),
            'La table metrics.csv contient aussi les cohortes disponibles propres a chaque combinaison. Comparer ces cohortes differentes exige de rapporter leur couverture. Une absence totale des modalites demandees produit une abstention.',
            'Les intervalles apparies des six modeles centraux sont dans AP_intervals.csv. Les 22 tetes exploratoires ne sont pas accompagnees ici d une preuve statistique corrigee pour comparaisons multiples.',
            '\n### RQ2',
            'Statut : **'+str(temporal_result.get('status'))+'**. '+str(temporal_result.get('reason','')),
            'Les details sont dans le rapport chronologique distinct. Le corpus a deja ete explore : ce test demeure interne.',
            '\n### RQ3', table(benchmark_result.groupby(['engine','status'], dropna=False).size().reset_index(name='measurements')),
            'Temps de demarrage Spark separe, cache OS non controle, execution locale sans cluster. Une mesure Spark absente ou bloquee ne constitue pas un benchmark Spark execute.',
            '\n### Diagnostics',
            'Voir feature_audit/ pour associations, collinearite, disponibilite, permutation technique et perturbation HTTP/HTTPS sur validation. Les effets de permutation ne prouvent pas une causalite.',
            'Voir error_review.csv pour les erreurs par scan et leur contexte. Ne pas reajuster seuils/features sur ces erreurs de test.',
            '\n## 6. Deployment et limites',
            'Prototype predict_scan/predict_url local, sans telechargement ni execution de page. Les scores ne sont pas certifies comme probabilites calibrees.',
            'Biais de source, plateformes et periodes, semantique des captures indisponibles, quasi-doublons non exacts et qualite des labels restent des limites. Prevoir revue humaine et nouvelle collecte equilibree.',
            'Validation finale requise : nouveau lot jamais utilise pour choisir modele, features, calibration ou seuil. En mode SMOKE_TEST, toutes les conclusions de performance restent interdites.'
        ]
        path = run/'PROJECT_FINAL_CRISP_DM.md'
        path.write_text('\n\n'.join(text)+'\n', encoding='utf-8')
        return path

    final_report_path = export_crisp_dm_report(context, temporal_result, benchmark_result)
    print('Rapport CRISP-DM issu des resultats reels :', final_report_path)


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    assert len(COMBO_SPECS) == 11
    assert len(combo_results) == 44 and combo_results.model.nunique() == 22
    assert len(all_28) == 28 and all_28.model.nunique() == 28
    assert url_model_text('http://example.org/login?q=1') == url_model_text('https://example.org/login?q=1')
    assert comparison.model.tolist()==list(MODEL_IDS)
    assert comparison.n.nunique()==1
    check_partitions(context['frame'])
    assert not encoded_collisions(context['frame'],context['preprocess'],settings)
    assert context['preprocess']['fitted_ids']==context['frame'].loc[context['frame'].split.eq('train_encoder')].scan_id.tolist()
    validate_notebook_session(_NOTEBOOK_SESSION)
    print('FINAL_AUDIT_RESULT='+json.dumps(dict(notebook_file=str(NOTEBOOK_FILE),models=comparison.model.tolist(),test_size=int(comparison.n.iloc[0]),
        temporal_status=temporal_result['status'],source_signature=source_signature(),mode=settings.mode,
        raw_rows=context['provenance']['raw_rows'],eligible_rows=context['provenance']['eligible_rows'],
        quarantined_rows=context['provenance']['quarantined_rows'],run=str(context['run']))))


## Partie B — analyse scientifique complète des28configurations

Le code d'analyse est également intégré ici : vérification des métriques et identifiants, disponibilité,65différences appariées d'AP avec200bootstraps par groupes, erreurs par contexte, revue humaine, faisabilité temporelle et figures finales. Il ne réentraîne aucun modèle.

En FULL, il travaille sur context['run'] produit par les cellules précédentes et écrit dans le sous-dossier analysis/ de ce run. En PRESENTATION, il est ignoré : les résultats déjà calculés sont rechargés dans la partie C.


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    import csv
    import pyarrow.parquet as pq
    import pyarrow.compute as pc
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    CENTRAL = ('U1','U2','U3','U4','M1','M2')
    MODALITIES = ('URL','HTML','IMAGE','TECH')
    LABELS = dict(U1='CNN URL', U2='CNN HTML', U3='CNN image', U4='MLP technique', M1='M1 · moyenne', M2='M2 · embeddings')
    COLORS = ['#4677a8','#24a79b','#d69552','#8078bb','#14a09a','#17375e']
    def read(path):
        return json.loads(Path(path).read_text(encoding='utf-8'))


    def write(path, value):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


    def sha(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as stream:
            for b in iter(lambda:stream.read(2**20),b''): h.update(b)
        return h.hexdigest()


    def frame(path):
        return pd.read_csv(path, float_precision='round_trip', dtype={'scan_id':str})


    def markdown(data):
        def fmt(v):
            if v is None or (isinstance(v,(float,np.floating)) and not np.isfinite(v)): return '—'
            return (f'{v:.4f}' if isinstance(v,(float,np.floating)) else str(v)).replace('|','/')
        return '\n'.join(['| '+' | '.join(data.columns)+' |','| '+' | '.join(['---']*len(data.columns))+' |']+
                         ['| '+' | '.join(fmt(v) for v in row)+' |' for row in data.itertuples(index=False,name=None)])


    def metrics(y,p,t):
        tn,fp,fn,tp=map(int,confusion_matrix(y,p>=t,labels=[0,1]).ravel())
        return dict(n=len(y),prevalence=float(np.mean(y)),AP=float(average_precision_score(y,p)),
                    ROC_AUC=float(roc_auc_score(y,p)),precision=tp/(tp+fp) if tp+fp else 0.,
                    recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),FPR=fp/(tn+fp),FNR=fn/(tp+fn),
                    TN=tn,FP=fp,FN=fn,TP=tp,threshold=t)


    def savefig(fig,name):
        fig.savefig(FIG/(name+'.png'),dpi=220,bbox_inches='tight',facecolor='white')
        fig.savefig(FIG/(name+'.pdf'),bbox_inches='tight')
        plt.close(fig)


    def finalize_current_run(context):
        global RUN, OUT, FIG
        RUN = context['run']
        OUT = RUN / 'analysis'
        FIG = OUT / 'reports/figures'

        start=time.perf_counter(); OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
        (OUT/'reports').mkdir(exist_ok=True)
        inventory=[]; preserved={}
        required=['comparison.csv','predictions.csv','combinations/metrics.csv','combinations/all_28_comparable_models.csv',
                  'combinations/predictions.csv','combinations/calibration_policies.csv','AP_intervals.csv',
                  'bd2/measurements.csv','bd2/scope.json','PROJECT_FINAL_CRISP_DM.md','preparation.json','preprocess.json',
                  'frozen.json','combinations/frozen.json','evaluation.json','combinations/study.json',
                  'figures/precision_recall.png','figures/confusion.png','figures/learning.png']
        required += [f'{model}/{name}' for model in ('U1','U2','U3','U4','M2') for name in ('best.pt','complete.json','history.csv')]
        required += [p.relative_to(RUN).as_posix() for p in (RUN/'feature_audit').glob('*.csv')]
        for name in required:
            p=RUN/name
            row=dict(path=str(p.relative_to(ROOT)),status='PASS' if p.is_file() else 'NOT_AVAILABLE')
            if p.is_file():
                row.update(bytes=p.stat().st_size,sha256=sha(p));preserved[row['path']]=row['sha256']
                if p.suffix=='.csv':
                    with p.open(encoding='utf-8-sig',newline='') as stream:
                        reader=csv.reader(stream);row['columns']=','.join(next(reader))
            inventory.append(row)
        pd.DataFrame(inventory).to_csv(OUT/'artifact_inventory.csv',index=False)
        assert all(r['status']=='PASS' for r in inventory),'Required artifacts missing'
        for manifest,base,key in [('frozen.json',RUN,'artifacts'),('evaluation.json',RUN,'files'),
                ('combinations/frozen.json',RUN/'combinations','artifacts'),('combinations/study.json',RUN/'combinations','files')]:
            for name,expected in read(RUN/manifest)[key].items(): assert sha(base/name)==expected,(manifest,name)
        prep=read(RUN/'preparation.json'); frozen=read(RUN/'frozen.json'); combo=read(RUN/'combinations/frozen.json')
        assert prep['code']==frozen['code'] and prep['split_sha256']==sha(ROOT/prep['split_relpath'])
        for model in ('U1','U2','U3','U4','M2'):
            assert read(RUN/model/'complete.json')['sha256']==sha(RUN/model/'best.pt')
        canonical=ROOT/'data/processed'/prep['extraction']['version'][:20]/'canonical.parquet'
        assert sha(canonical)==prep['extraction']['canonical_sha256']
        split=pd.read_parquet(ROOT/prep['split_relpath']);assert split.scan_id.is_unique
        metadata_cols=['scan_id','label_id','source_name','provider','http_status','scan_created_at','site_id','normalized_url',
            'url_input_hash','html_input_hash','raw_html_hash','image_input_hash','html_state','image_state','uniform',
            'quality_state','primary_eligible','target_url','capture_errors','negative_duration']
        available=pq.ParquetFile(canonical).schema_arrow.names
        metadata_cols=[name for name in metadata_cols if name in available]
        raw=pd.read_parquet(canonical,columns=metadata_cols)
        eligible=raw.loc[raw.primary_eligible].merge(split,on=['scan_id','label_id'],validate='one_to_one')
        for name in ['group_id','site_id','normalized_url','url_input_hash','html_input_hash','raw_html_hash','image_input_hash','encoding_component']:
            if name not in eligible:continue
            selected=eligible.loc[eligible[name].notna() & eligible[name].ne('')]
            assert not selected.groupby(name).split.nunique().gt(1).any(),name
        source_csv=ROOT/'data/raw/phishark2026_public_metadata.csv'
        public=pd.read_csv(source_csv,low_memory=False)
        assert sha(source_csv)==prep['extraction']['binding']['metadata_sha256']
        labels=public.set_index('scan_id').label_id
        assert raw.set_index('scan_id').label_id.eq(labels.loc[raw.scan_id].set_axis(raw.scan_id)).all()
        # Derive complete-case cohort independently from captured inputs, without loading all HTML into Python.
        feature_names=read(RUN/'preprocess.json')['features'];mask_rows=[]
        cols=['scan_id','target_url','html_text','html_state','image_state',*feature_names]
        for batch in pq.ParquetFile(canonical).iter_batches(batch_size=2048,columns=cols):
            table=batch.to_pydict()
            tech=np.column_stack([np.asarray(table[name],float) for name in feature_names])
            mask=np.column_stack(([len(v or '')>0 for v in table['target_url']],
                [state in ('ok','truncated') and len(v or '')>0 for state,v in zip(table['html_state'],table['html_text'])],
                [state=='ok' for state in table['image_state']],np.isfinite(tech).any(axis=1)))
            mask_rows.extend(zip(table['scan_id'],mask.all(axis=1)))
        complete_map=dict(mask_rows)
        test=split.loc[split.split.eq('test')].sort_values('scan_id').reset_index(drop=True)
        common=test.loc[test.scan_id.map(complete_map)].copy().reset_index(drop=True)
        y=common.label_id.to_numpy();ids=common.scan_id.tolist();full_ids=test.scan_id.tolist()
        common[['scan_id','label_id','group_id']].to_csv(OUT/'common_cohort_ids.csv',index=False)
        central=frame(RUN/'predictions.csv');central_test=central.loc[central.split.eq('test')].copy()
        additional=frame(RUN/'combinations/predictions.csv');policies={r['model']:r for r in combo['policies']}
        scores={};thresholds={};rows=[];full_rows=[]
        stored=frame(RUN/'combinations/all_28_comparable_models.csv').set_index('model')
        for name in [*CENTRAL,*policies]:
            selected=(central_test if name in CENTRAL else additional).loc[lambda d:d.model.eq(name)].set_index('scan_id')
            assert selected.index.is_unique and set(selected.index)==set(full_ids),name
            selected=selected.loc[full_ids]
            assert np.array_equal(selected.label_id,test.label_id),name
            if name not in CENTRAL:
                observed=set(selected.loc[selected.complete_case,'scan_id']) if 'scan_id' in selected else set(selected.index[selected.complete_case])
                assert observed==set(ids)
                assert selected.loc[~selected.eligible,'probability'].isna().all()
            t=frozen['policies'][name]['primary'] if name in CENTRAL else policies[name]['threshold']
            assert t is not None
            thresholds[name]=t;scores[name]=selected.loc[ids,'probability'].to_numpy()
            assert np.isfinite(scores[name]).all()
            if name in CENTRAL:
                modality=MODALITIES[CENTRAL.index(name)] if name.startswith('U') else '+'.join(MODALITIES)
                method='UNIMODAL' if name.startswith('U') else 'LATE' if name=='M1' else 'MLP_EMBED'
            else:modality=policies[name]['modalities'];method=policies[name]['method']
            values=metrics(y,scores[name],t)
            for key in ['n','AP','ROC_AUC','precision','recall','F1','FPR','FNR','FP','FN','TN','TP']:
                assert np.isclose(values[key],stored.loc[name,key],rtol=0,atol=1e-8),(name,key,values[key],stored.loc[name,key])
            rows.append(dict(model=name,modalities=modality,fusion=method,modality_count=len(modality.split('+')),
                             coverage=len(common)/len(test),**values))
            if name in CENTRAL:
                values=metrics(test.label_id.to_numpy(),selected.probability.to_numpy(),t)
                full_rows.append(dict(model=name,coverage=1.,**values))
        main_table=pd.DataFrame(rows);main_table.to_csv(OUT/'table_28_common.csv',index=False)
        main_table.to_json(OUT/'table_28_common.json',orient='records',force_ascii=False,indent=2)
        pd.DataFrame(full_rows).to_csv(OUT/'six_models_full_test.csv',index=False)
        own=frame(RUN/'combinations/metrics.csv');own.loc[own.cohort.eq('available')].to_csv(OUT/'table_22_available.csv',index=False)
        # Independently recompute own-cohort metrics, including abstentions and calibration thresholds.
        for name,item in policies.items():
            selected=additional.loc[additional.model.eq(name)].set_index('scan_id').loc[full_ids]
            keep=selected.eligible.to_numpy(bool)&selected.probability.notna().to_numpy()
            values=metrics(selected.label_id.to_numpy()[keep],selected.probability.to_numpy()[keep],item['threshold'])
            old=own.loc[own.model.eq(name)&own.cohort.eq('available')].iloc[0]
            for key in ['n','AP','F1','FPR','FNR','FP','FN']:assert np.isclose(values[key],old[key],rtol=0,atol=1e-8),(name,key)
        print('All 28 common-cohort and 22 available-cohort metrics verified',flush=True)
        # Efficient paired site/content-group bootstrap. Ranking is fixed; multiplicities are sampled by group.
        names=list(scores);group_ids,inverse=np.unique(common.group_id,return_inverse=True);rng=np.random.default_rng(42)
        ranking={}
        for name,p in scores.items():
            order=np.argsort(-p,kind='stable');ends=np.r_[np.flatnonzero(np.diff(p[order])),len(p)-1]
            ranking[name]=(order,ends)
        boot=np.full((200,len(names)),np.nan)
        for repetition in range(200):
            weight=np.bincount(rng.integers(0,len(group_ids),len(group_ids)),minlength=len(group_ids))[inverse]
            positives=float(np.dot(weight,y))
            if positives==0 or np.dot(weight,1-y)==0:continue
            for j,name in enumerate(names):
                order,ends=ranking[name];tp=np.cumsum(weight[order]*y[order])[ends];total=np.cumsum(weight[order])[ends]
                precision=np.divide(tp,total,out=np.zeros_like(tp,dtype=float),where=total>0)
                boot[repetition,j]=np.dot(np.diff(np.r_[0.,tp])/positives,precision)
        index=main_table.set_index('model');pairs=[]
        requests=[(fusion,u,None) for fusion in ('M1','M2') for u in CENTRAL[:4]]+[('M2','M1',None)]
        from itertools import combinations
        for method in ('LATE','LINEAR_EMBED'):
            for count in (2,3,4):
                for choice in combinations(MODALITIES,count):
                    target=method+'__'+'+'.join(choice)
                    for added in choice:
                        reduced=tuple(v for v in choice if v!=added)
                        baseline='U'+str(MODALITIES.index(reduced[0])+1) if len(reduced)==1 else method+'__'+'+'.join(reduced)
                        requests.append((target,baseline,added))
        for a,b,added in requests:
            difference=boot[:,names.index(a)]-boot[:,names.index(b)];valid=difference[np.isfinite(difference)]
            lo,hi=np.quantile(valid,[.025,.975])
            pairs.append(dict(model=a,reference=b,added_modality=added,AP_difference=index.loc[a,'AP']-index.loc[b,'AP'],
                lower95=lo,upper95=hi,valid_replicates=len(valid),unit='connected_site_content_group',cohort_n=len(common),
                multiple_comparisons_corrected=False,interpretation='exploratory_internal'))
        paired=pd.DataFrame(pairs);paired.to_csv(OUT/'paired_modality_contributions.csv',index=False)
        write(OUT/'bootstrap_scope.json',dict(repetitions=200,seed=42,groups=len(group_ids),cohort_n=len(common),paired=True,
            unit='original_connected_site_content_group',selection_on_test=False,multiplicity_adjusted=False))
        # Frozen operating points only; no new threshold is selected from the test.
        operating=[]
        for name in names:
            policies_for_name=frozen['policies'][name] if name in CENTRAL else dict(primary=policies[name]['threshold'],recall85=policies[name]['threshold_recall85'])
            for policy,t in policies_for_name.items():
                if t is not None:operating.append(dict(model=name,policy=policy,threshold_origin='calibration',cohort='common',**metrics(y,scores[name],t)))
        pd.DataFrame(operating).to_csv(OUT/'operating_points_frozen.csv',index=False)
        # Errors on full test for central models: denominators by class retained for every stratum.
        context=raw.set_index('scan_id').loc[full_ids].reset_index();context['month']=pd.to_datetime(context.scan_created_at,utc=True,format='mixed').dt.strftime('%Y-%m')
        error_rows=[];examples=[]
        for model in CENTRAL:
            part=central_test.loc[central_test.model.eq(model)].set_index('scan_id').loc[full_ids].reset_index()
            part=part.merge(context.drop(columns='label_id'),on='scan_id',validate='one_to_one')
            part['prediction']=part.probability.ge(thresholds[model]).astype(int)
            part['error']=np.where(part.label_id.eq(0)&part.prediction.eq(1),'FP',np.where(part.label_id.eq(1)&part.prediction.eq(0),'FN','correct'))
            for dim in ('provider','http_status','source_name','month','html_state','image_state','uniform','quality_state'):
                for value,sub in part.groupby(dim,dropna=False):
                    negatives=int(sub.label_id.eq(0).sum());positives=int(sub.label_id.eq(1).sum())
                    fp=int(sub.error.eq('FP').sum());fn=int(sub.error.eq('FN').sum())
                    error_rows.append(dict(model=model,dimension=dim,value=str(value),n=len(sub),benign=negatives,phishing=positives,
                        FP=fp,FN=fn,FPR=fp/negatives if negatives else None,FNR=fn/positives if positives else None))
            for error in ('FP','FN'):
                chosen=part.loc[part.error.eq(error)].sort_values(['probability','scan_id'],ascending=[error=='FN',True]).head(4)
                for row in chosen.to_dict('records'):
                    examples.append({k:row.get(k) for k in ['scan_id','label_id','model','probability','error','provider','http_status','scan_created_at','source_name','html_state','image_state','uniform','quality_state','target_url']})
        pd.DataFrame(error_rows).to_csv(OUT/'error_strata.csv',index=False)
        queue=pd.DataFrame(examples);queue['review_status']='PENDING_HUMAN';queue['label_changed']=False;queue.to_csv(OUT/'manual_error_review_queue.csv',index=False)
        # Temporal feasibility replay: same raw-date quantiles and same raw identity components, no model fitting.
        source=raw.loc[raw.primary_eligible].copy();source['group_id']=connected_groups(source)
        dates=pd.to_datetime(source.scan_created_at,format='mixed',utc=True);edges=[dates.quantile(float(q)) for q in np.cumsum(prep['settings']['split_fractions'])[:-1]]
        roles=('train_encoder','train_fusion','validation','calibration','test')
        source['role']=[roles[int(np.searchsorted(edges,date,side='right'))] for date in dates]
        crossing=source.groupby('group_id').role.nunique().gt(1);source['purged']=source.group_id.isin(crossing.index[crossing])
        source.groupby(['role','purged','label_id']).size().reset_index(name='n').to_csv(OUT/'temporal_class_counts_before_after_purge.csv',index=False)
        for period in ('D','M'):
            public.assign(period=pd.to_datetime(public.scan_created_at,format='mixed',utc=True).dt.strftime('%Y-%m-%d' if period=='D' else '%Y-%m')).groupby(['period','label_id','source_name']).size().reset_index(name='n').to_csv(OUT/f'temporal_distribution_{period}.csv',index=False)
        kept=source.loc[~source.purged];feasibility={role:kept.loc[kept.role.eq(role)].label_id.value_counts().to_dict() for role in roles}
        write(OUT/'temporal_feasibility.json',dict(status='NOT_FEASIBLE' if any(len(v)!=2 for v in feasibility.values()) else 'FEASIBLE',
            boundaries_utc=list(map(str,edges)),counts=feasibility,purged=int(source.purged.sum()),original_status=read(ROOT/'artifacts/temporal_full_experiment.json'),
            date_modified=False,labels_modified=False,networks_fitted=False))
        # Five publication-ready main figures; every model figure uses the verified same cohort.
        plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.spines.top':False,'axes.spines.right':False})
        fig,ax=plt.subplots(figsize=(9.5,4.8));values=index.loc[list(CENTRAL),'AP'];ax.bar([LABELS[n] for n in CENTRAL],values,color=COLORS)
        ax.set(ylim=(0,1.06),ylabel='Average precision (AP)',title=f'Comparaison centrale · mêmes {len(common):,} scans complets'.replace(',',' '))
        ax.axhline(y.mean(),ls='--',color='#9ca3af',label=f'Prévalence phishing : {y.mean():.3f}');ax.legend(fontsize=9)
        for i,v in enumerate(values):ax.text(i,v+.015,f'{v:.3f}',ha='center')
        savefig(fig,'01_modeles_AP')
        heat=main_table.loc[main_table.model.str.contains('__')].pivot(index='modalities',columns='fusion',values='AP').reindex(columns=['LATE','LINEAR_EMBED'])
        fig,ax=plt.subplots(figsize=(8.5,8));im=ax.imshow(heat.values,cmap='YlGnBu',vmin=.5,vmax=1,aspect='auto');ax.set_xticks([0,1],['Moyenne des probabilités','Embeddings + logistique']);ax.set_yticks(range(len(heat)),heat.index)
        for i in range(len(heat)):
            for j in range(2):ax.text(j,i,f'{heat.iloc[i,j]:.4f}',ha='center',va='center',color='white' if heat.iloc[i,j]>.86 else '#10243c')
        fig.colorbar(im,ax=ax,label='AP');ax.set_title('11 sous-ensembles × 2 fusions · cohorte commune');savefig(fig,'02_combinaisons_heatmap')
        fig,ax=plt.subplots(figsize=(10,6));markers={'UNIMODAL':'s','LATE':'o','MLP_EMBED':'*','LINEAR_EMBED':'^'}
        for method,part in main_table.groupby('fusion'):
            ax.scatter(part.FPR*100,part.recall*100,s=35+part.modality_count*23,marker=markers[method],label=method,alpha=.8,edgecolors='white')
        for name in ['U1','U2','U3','U4','M1','M2','LINEAR_EMBED__URL+HTML+TECH']:
            row=index.loc[name];ax.annotate(name.replace('LINEAR_EMBED__','Linéaire '),(row.FPR*100,row.recall*100),xytext=(6,5),textcoords='offset points',fontsize=8)
        ax.set(xlabel='Faux positifs parmi les bénins — FPR (%)',ylabel='Rappel phishing (%)',title='Détection et fausses alertes · seuils figés sur calibration');ax.grid(alpha=.15);ax.legend(fontsize=9,loc='upper center',bbox_to_anchor=(.5,-.16),ncol=4);savefig(fig,'03_compromis_FPR_rappel')
        fig,ax=plt.subplots(figsize=(10,6));chosen=[*CENTRAL,'LINEAR_EMBED__URL+HTML+TECH','LATE__URL+HTML']
        pr_payload=[]
        for name in chosen:
            precision,recall,_=precision_recall_curve(y,scores[name]);ax.step(recall,precision,where='post',label=f'{name} · AP={index.loc[name,"AP"]:.3f}',lw=1.6)
            positions=np.unique(np.linspace(0,len(recall)-1,150,dtype=int));pr_payload.append(dict(model=name,recall=recall[positions].tolist(),precision=precision[positions].tolist()))
        ax.axhline(y.mean(),ls='--',color='#9ca3af',label=f'Prévalence : {y.mean():.3f}');ax.set(xlabel='Rappel phishing',ylabel='Précision phishing',xlim=(0,1),ylim=(0,1.02),title='Courbes précision–rappel · comparaison interne exploratoire');ax.legend(fontsize=8,ncol=2);savefig(fig,'04_precision_recall')
        fig,axes=plt.subplots(2,3,figsize=(11,7))
        for ax,name in zip(axes.flat,CENTRAL):
            row=index.loc[name];matrix=np.array([[row.TN,row.FP],[row.FN,row.TP]],int);ax.imshow(matrix,cmap='Blues',vmin=0,vmax=max(matrix.ravel()))
            for (i,j),v in np.ndenumerate(matrix):ax.text(j,i,f'{["TN","FP","FN","TP"][2*i+j]}\n{v:,}'.replace(',',' '),ha='center',va='center',color='white' if v>matrix.max()*.6 else '#10243c',fontsize=12)
            ax.set_xticks([0,1],['Prédit bénin','Prédit phishing']);ax.set_yticks([0,1],['Réel bénin','Réel phishing']);ax.set_title(LABELS[name])
        fig.suptitle(f'Matrices de confusion · {len(common):,} scans communs'.replace(',',' '));fig.tight_layout();savefig(fig,'05_confusion_matrices')
        # Measured pandas results and explicit Spark resource status are kept separate.
        pandas=frame(RUN/'bd2/measurements.csv');pandas=pandas.loc[pandas.status.eq('MEASURED')]
        summary=pandas.groupby(['engine','scenario','n'],dropna=False).agg(repetitions=('seconds','size'),seconds_median=('seconds','median'),seconds_min=('seconds','min'),seconds_max=('seconds','max'),
            throughput_median=('scans_per_second','median'),peak_rss_bytes=('peak_process_tree_rss_bytes','max'),input_bytes=('input_bytes','median'),output_bytes=('output_bytes','median'),data_files=('data_files','median')).reset_index()
        summary.to_csv(OUT/'benchmark_summary.csv',index=False)
        benchmark_records=summary.to_dict('records')
        for field,ylabel,name in [('seconds_median','Temps médian mesuré (s)','06_bigdata_temps'),('throughput_median','Débit médian (scans/s)','07_bigdata_debit')]:
            fig,ax=plt.subplots(figsize=(8,4.5))
            for scenario,part in summary.groupby('scenario'):ax.plot(part.n,part[field],marker='o',label='pandas/PyArrow · '+scenario)
            ax.set(xlabel='Nombre de scans',ylabel=ylabel,title='Lecture → validation → cast → Parquet → relecture vérifiée');ax.legend(fontsize=8);ax.grid(alpha=.15);savefig(fig,name)
        # Bind all derived tables and visuals to current-run immutable inputs.
        write(OUT/'preserved_run_hashes.json',preserved)
        model_training=[read(RUN/name/'complete.json') for name in ('U1','U2','U3','U4','M2')]
        summary=dict(run=str(RUN.relative_to(ROOT)),code=prep['code'],raw_rows=prep['raw_rows'],eligible_rows=prep['eligible_rows'],
            quarantined_rows=prep['quarantined_rows'],test_n=len(test),common_n=len(common),coverage=len(common)/len(test),
            common_prevalence=float(y.mean()),test_prevalence=float(test.label_id.mean()),group_count=len(group_ids),
            metrics_verified=28,available_metrics_verified=22,bootstrap_repetitions=200,networks_trained=False,
            label_changes=False,threshold_changes=False,selection_on_test=False,seconds=time.perf_counter()-start,
            training=model_training,class_counts=prep['class_counts'],extraction_seconds=prep['extraction']['seconds'])
        write(OUT/'analysis_summary.json',summary)
        write(OUT/'deck_data.json',dict(summary=summary,models=main_table.to_dict('records'),six_full=full_rows,
            heatmap=dict(rows=heat.index.tolist(),columns=heat.columns.tolist(),values=heat.values.tolist()),
            pr_curves=pr_payload,benchmark=benchmark_records))
        # Report comparisons most useful for oral interpretation, with no winner selected.
        text='# Analyse appariée des modalités\n\nMême cohorte : '+str(len(common))+' scans. 200 bootstraps par groupes ; comparaisons exploratoires, sans correction de multiplicité.\n\n'
        text+=markdown(paired.loc[paired.added_modality.eq('IMAGE')|paired.model.isin(['M1','M2']),['model','reference','AP_difference','lower95','upper95']])
        (OUT/'reports/ANALYSE_MODALITES.md').write_text(text+'\n',encoding='utf-8')
        print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)

        central_table = pd.DataFrame(rows)
        analysis_report = [
            '# Analyse finale des 28 configurations — résultats de cette exécution',
            f"Cohorte commune : {len(common)} scans ; test complet : {len(test)} ; couverture : {len(common)/len(test):.4f}.",
            'Les seuils sont issus de calibration. Aucun modèle final n’est sélectionné sur le test déjà exploré.',
            markdown(central_table[['model','modalities','fusion','AP','ROC_AUC','precision','recall','F1','FPR','FNR','FP','FN','coverage']]),
            '## Incertitude appariée — 200 bootstraps par groupes, comparaisons exploratoires',
            markdown(paired[['model','reference','added_modality','AP_difference','lower95','upper95']]),
            '## Limites',
            'Les associations source/classe et période/classe peuvent créer des proxies. Les contrôles exacts ne prouvent pas l’absence de tous les quasi-doublons.',
            'RQ2 : consulter le résultat chronologique de cette exécution ; aucune métrique n’est inventée si le protocole est NOT_FEASIBLE.',
            'RQ3 : seuls les moteurs réellement mesurés figurent dans les résultats. Un blocage Spark n’est pas un temps nul.',
            'Validation externe et collecte équilibrée nécessaires avant toute certification de déploiement.'
        ]
        (OUT/'reports/ANALYSE_FINALE_28.md').write_text('\n\n'.join(analysis_report)+'\n', encoding='utf-8')
        return summary


In [ ]:
if MODE == 'FULL_EXPERIMENT':
    final_analysis = finalize_current_run(context)
    print('Analyse scientifique complète enregistrée :', context['run'] / 'analysis')


## Partie C — restitution des résultats FULL déjà exécutés

Cette partie fonctionne en PRESENTATION. Elle ne dépend d'aucun autre notebook ou code du projet : les fonctions de lecture sont intégrées ci-dessous. Les résultats affichés ont leur propre provenance enregistrée, distincte de la signature actuelle du fichier.


In [ ]:
if MODE == 'PRESENTATION':
    RUN = ROOT / 'artifacts/runs/full_experiment/grouped/77162f55317be0844ec3'
    OUT = ROOT / 'artifacts/published'
    def read(path):return json.loads(Path(path).read_text(encoding='utf-8'))


    def sha(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda:stream.read(2**20),b''):h.update(block)
        return h.hexdigest()


    def load_evidence():
        import pandas as pd
        preserved=read(OUT/'manifest.json')['files']
        for name,expected in preserved.items():
            if sha(OUT/name)!=expected:raise ValueError('Artifact changed: '+name)
        tables={name:pd.read_csv(OUT/file) for name,file in dict(common='table_28_common.csv',full='six_models_full_test.csv',
            available='table_22_available.csv',contributions='paired_modality_contributions.csv',errors='error_strata.csv',
            operating='operating_points_frozen.csv',benchmark='benchmark_summary.csv').items()}
        if len(tables['common'])!=28 or tables['common'].n.nunique()!=1:raise ValueError('Invalid common cohort')
        return dict(summary=read(OUT/'analysis_summary.json'),tables=tables,
            temporal=read(OUT/'temporal_feasibility.json'),spark=read(OUT/'spark_status.json'),run=RUN)


    def saved_demo(evidence,scan_id=None):
        """Recorded test prediction, clearly not a live prediction of a new website."""
        import pandas as pd
        ledger=pd.read_csv(OUT/'demo_predictions.csv',dtype={'scan_id':str})
        test=ledger
        scan_id=scan_id or sorted(test.scan_id.unique())[0]
        result=test.loc[test.scan_id.eq(scan_id),['scan_id','model','label_id','probability']].copy()
        if result.empty:raise ValueError('Unknown test scan ID')
        frozen=read(ROOT/'models/frozen.json')
        result['threshold']=result.model.map({name:p['primary'] for name,p in frozen['policies'].items()})
        result['predicted_phishing']=result.probability.ge(result.threshold)
        result['scope']='Recorded FULL test inference; not a new live capture'
        return result


    from IPython.display import display, Markdown, Image
    evidence = load_evidence()
    tables = evidence['tables']
    print('Résultats FULL vérifiés ; code producteur :', evidence['summary']['code'])
    print('Aucun réseau chargé ou entraîné en PRESENTATION.')


## 1. Business Understanding
**Objectif :** détecter la réputation phishing d'une URL à partir de quatre modalités, en distinguant réputation annotée et contenu effectivement capturé. Un HTTP404 ne devient pas automatiquement bénin.

**RQ1 :** la fusion surpasse-t-elle les unimodaux ? **RQ2 :** généralise-t-elle dans le temps ? **RQ3 :** quel coût de calcul selon le volume et le moteur ?


## 2. Data Understanding
**Objectif :** connaître les observations et les biais de collecte avant d'interpréter les scores.
Le corpus comporte 67 502 scans, 34 115 bénins et 33 387 phishing. Les sources sont fortement associées aux classes ; ce benchmark déjà exploré n'est pas une validation externe indépendante.


In [ ]:
if MODE == 'PRESENTATION':
    display({k:evidence['summary'][k] for k in ['raw_rows','eligible_rows','quarantined_rows','test_n','common_n','coverage','common_prevalence']})


**Interprétation :** une archive discordante est conservée en quarantaine. Les 28 configurations sont comparées sur les mêmes 13 212 scans complets ; les six modèles ont aussi un registre de test complet distinct de 13 439 scans.


## 3. Data Preparation et Big Data
**Objectif :** construire des entrées alignées et des groupes indépendants.

CSV/ZIP originaux → JSON autorisés, HTML visible, images96×96 → Parquet → groupes sites/locataires/contenus/encodages → cinq rôles disjoints. Imputation et vocabulaires ajustés sur train_encoder ; fusion sur train_fusion ; validation choisit les checkpoints ; calibration les seuils ; test mesure les performances.


In [ ]:
if MODE == 'PRESENTATION':
    import pandas as pd
    display(pd.DataFrame(evidence['summary']['class_counts']))


## 4. Modeling
**Objectif :** mesurer la complémentarité des modalités.

U1 CNN1D URL sans HTTP/HTTPS ; U2 CNN1D HTML visible ; U3 CNN2D screenshots ; U4 MLP technique21variables+masques. Chaque encodeur produit32valeurs.

M1 moyenne les probabilités disponibles. M2 apprend un MLP sur embeddings gelés+masques. Les11 sous-ensembles (6paires,4triplets,1quadruplet) sont étudiés par moyenne et logistique sur embeddings :22 études supplémentaires, sans22 nouveaux CNN.


In [ ]:
if MODE == 'PRESENTATION':
    display(pd.DataFrame(evidence['summary']['training'])[['model','best_epoch','validation_AP','train_seconds','parameters','device']])


## 5. Evaluation — comparaison équitable
**Objectif :** comparer AP, erreurs et couverture sur les mêmes scans, aux seuils déjà gelés.
Les tableaux ci-dessous sont recalculés depuis les prédictions enregistrées. Aucun classement de test ne sert à choisir rétroactivement un modèle final.


In [ ]:
if MODE == 'PRESENTATION':
    display(tables['common'][['model','modalities','fusion','n','AP','ROC_AUC','precision','recall','F1','FPR','FNR','FP','FN','coverage']])


In [ ]:
if MODE == 'PRESENTATION':
    display(Image(filename=str(ROOT/'reports/figures/01_modeles_AP.png')))
    display(Image(filename=str(ROOT/'reports/figures/02_combinaisons_heatmap.png')))


### Contribution des modalités
Les différences appariées utilisent200bootstraps par groupes sites/contenus. Elles restent exploratoires et non corrigées pour comparaisons multiples. Les images sont peu discriminantes seules ; certaines fusions pâtissent de leur ajout. Les différences observées ne prouvent pas une cause liée au contenu visuel sans revue dédiée.


In [ ]:
if MODE == 'PRESENTATION':
    display(tables['contributions'].loc[tables['contributions'].added_modality.eq('IMAGE')|tables['contributions'].model.isin(['M1','M2']),['model','reference','AP_difference','lower95','upper95']])


### Faux positifs, phishing manqués et seuils
M1 privilégie davantage le rappel ; M2 réduit fortement les faux positifs mais manque davantage de phishing. AP décrit un classement, pas le compromis d'un seuil unique. Les points primary et recall85 proviennent de calibration ; aucune optimisation sur test.


In [ ]:
if MODE == 'PRESENTATION':
    display(Image(filename=str(ROOT/'reports/figures/03_compromis_FPR_rappel.png')))
    display(Image(filename=str(ROOT/'reports/figures/04_precision_recall.png')))
    display(Image(filename=str(ROOT/'reports/figures/05_confusion_matrices.png')))


In [ ]:
if MODE == 'PRESENTATION':
    display(tables['operating'].loc[tables['operating'].model.isin(['M1','M2']),['model','policy','threshold','precision','recall','FPR','FP','FN']])
    display(tables['available'][['model','n','AP','coverage']])


### RQ2 — généralisation temporelle
**La généralisation temporelle n'a pas pu être démontrée avec le corpus actuel.** Après les frontières temporelles et la purge des groupes chevauchants, calibration contient des phishing mais aucun bénin. Les dates et labels ne sont pas réassignés pour créer un score artificiel. Une collecte équilibrée dans chaque période et un nouveau lot indépendant sont nécessaires.


In [ ]:
if MODE == 'PRESENTATION':
    display(evidence['temporal'])


### RQ3 — mesures pandas/PySpark
24 mesures pandas/PyArrow existent (deux scénarios, quatre fractions, trois répétitions). Spark a été vérifié dans un processus séparé sans PyTorch ; BLOCKED_RESOURCE si la mémoire disponible est insuffisante. Aucun temps Spark n'est inventé. L'exécution locale n'est pas un cluster distribué.


In [ ]:
if MODE == 'PRESENTATION':
    display(tables['benchmark'])
    display({k:evidence['spark'][k] for k in ['status','reason','measured_spark_times'] if k in evidence['spark']})
    display(Image(filename=str(ROOT/'reports/figures/06_bigdata_temps.png')))
    display(Image(filename=str(ROOT/'reports/figures/07_bigdata_debit.png')))


## 6. Deployment — démonstration locale
**Objectif :** expliquer l'utilisation des scores et des seuils, sans accès réseau.
La démonstration ci-dessous relit une inférence réelle du test : ce n'est pas une capture nouvelle ni une preuve de phishing actif. Les fonctions predict_scan et predict_url du notebook FULL préservé permettent l'inférence de nouveaux inputs avec les checkpoints gelés.


In [ ]:
if MODE == 'PRESENTATION':
    display(saved_demo(evidence))


### Conclusion défendable
**RQ1 :** gain d'AP interne pour les fusions centrales par rapport aux unimodaux, avec compromis rappel/FPR à préciser. **RQ2 :** non démontrée. **RQ3 :** coût pandas mesuré, comparaison Spark bloquée par les ressources.

Les limites demeurent : biais sources/périodes, proxy de collecte, CNN image faible, réputation distincte du contenu, test déjà exploré et absence de lot externe scellé. Les modèles et seuils restent figés pour une future évaluation indépendante.


In [ ]:
if MODE == 'PRESENTATION':
    print('PRESENTATION_VALIDATION_OK',len(tables['common']),evidence['summary']['common_n'])
